# BetaTour v15r2 — canlı aday değiştirme pilotu

LKH-3.0.14'ün POPMUSIC-6 aday listesiyle ilk denemesi tamamlandığında, en çok düğümlerin %5'inde düşük öncelikli bir adayı başka bir adayla **yerinde değiştiriyoruz**. BetaTour, en yakın yeni komşu ve deterministik rastgele seçim aynı kaynak düğümleri ve aynı çıkarılacak adayları kullanır. İlk altı turun bütün kenarları ve her aday satırının ilk iki girdisi korunur. Native POPMUSIC-6 kesintisiz pratik kıyastır; sham aynı duraklamayı ve model çıkarımını öder.

Bu **daha önce incelenmiş dört 10.000 düğümlü örnek üzerinde keşif pilotudur**. Sonuçlar bir üstünlük iddiası veya bağımsız doğrulama değildir. Bir politika gerçekten umut verirse, kodu ve analiz planını dondurup yeni örneklerde tekrar test edeceğiz. Her kolun toplam duvar saati sınırına ilk tur, bekleme, model çıkarımı ve sonraki arama dahildir. Bir örnekteki beş kol sırayla, her kolda altı LKH süreci paralel çalışır.

Colab Pro'da CPU, GPU veya High RAM çalışma ortamı seçebilirsiniz. CUDA varsa model onu kullanır; yoksa CPU kullanır. CPU'da veya 6'dan az vCPU ile her kol **300 saniye**; en az 6 vCPU'lu GPU'da **120 saniye** çalışır. Bütçe ve donanım profili Drive deney adına ve manifestine yazılır. Oturum kesilirse hücreleri yeniden çalıştırın: tamamlanan kollar doğrulanıp atlanır. Aynı Drive deneyi farklı donanım profiliyle sürdürülmez. Pilot tamamlandığında sonuç ZIP'i tarayıcıya otomatik indirilir; ham denetim ZIP'i Drive'da kalır.

Yaklaşık süre: GPU/6+ vCPU için 50–55 dakika; CPU veya az vCPU için yaklaşık 110 dakika ve kurulum. Her kol eşit bütçe alır. Süre yetmezse başarısız denemenin günlükleri Drive'a kaydedilir; eksik kol sonuç karşılaştırmasına girmez.


In [ ]:
EXPERIMENT_STEM = "v15r2_live_swap_pilot"
GPU_BUDGET_S = 120.0
CPU_OR_LOW_VCPU_BUDGET_S = 300.0
SEEDS = list(range(301, 307))
ARMS = ["native_pop6", "sham", "random_swap", "nearest_swap", "beta_swap"]
INSTANCE_SPECS = [
    ["uni10k", 10000, 2, "uniform", 4107, 120.0],
    ["clu10k", 10000, 2, "clustered", 4208, 120.0],
]
EXPECTED_XY_SHA256 = {
    "augment_uni10k_000": "0d5d680b465faa68d33467c4f70bb5ad1e52b25a2d444066ab922130458efcd6",
    "augment_uni10k_001": "00747175507524f481bd6e34ea55f0a5474d3dcd642a67543d871d9a64b53796",
    "augment_clu10k_000": "072fc96ec668bb9c95b3a52562f23cedb0b2e89494946d159243b638b7b3dc8e",
    "augment_clu10k_001": "965fe4315060118d313a357404f750a23ea89cc11c174cb1138a9f32b312ca22",
}
MAX_NEW_ARMS = None  # Örneğin 2: oturumda iki yeni kol çalıştırıp sonra devam edin.
SWAP_FRACTION = 0.05
GRAPH_K = 16
REPO_COMMIT = "aac51871a50df33daba2c0492cca2fe8f0c31089"
CHECKPOINT_SHA256 = "2322f66c752913f6179f8c61f6b50b0f9d21426857d3ec0155e02891fb3ebd70"
LKH_TAR_SHA256 = "4c4b9e6a77e50a1eb1eb2c8cb384ed5ef64e0dc88fdbf37bd693798469e1cf33"
LKH_FINDTOUR_SHA256 = "567e72acc9b7392488dd158be4df0267560424204a0e88d2d9b62f923a7c0eee"
assert len(SEEDS) == 6 and len(EXPECTED_XY_SHA256) == 4


In [ ]:
import base64, hashlib, importlib.util, json, math, os, platform, shutil, subprocess, sys, tarfile, time, urllib.request, uuid, zipfile
from pathlib import Path

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
for package in ("scipy", "pandas"):
    if importlib.util.find_spec(package) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)
import numpy as np
import scipy
import pandas as pd
import torch
from IPython.display import display

torch.set_num_threads(1)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "none"
CPU_COUNT = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else (os.cpu_count() or 1)
cpu_max = Path("/sys/fs/cgroup/cpu.max")
if cpu_max.exists():
    quota, period = cpu_max.read_text().split()[:2]
    if quota != "max":
        CPU_COUNT = min(CPU_COUNT, max(1, math.floor(int(quota) / int(period))))
CPU_COUNT = max(1, CPU_COUNT)
def cpu_model_name():
    info = Path("/proc/cpuinfo")
    if info.exists():
        for line in info.read_text().splitlines():
            if line.lower().startswith("model name"):
                return line.split(":", 1)[1].strip()
    return platform.processor() or "unknown"

PER_ARM_BUDGET_S = (CPU_OR_LOW_VCPU_BUDGET_S
                    if DEVICE.type == "cpu" or CPU_COUNT < 6 else GPU_BUDGET_S)
EXPERIMENT_NAME = f"{EXPERIMENT_STEM}_{DEVICE.type}{int(PER_ARM_BUDGET_S)}"
INSTANCE_SPECS = [list(spec[:5]) + [PER_ARM_BUDGET_S] for spec in INSTANCE_SPECS]
WORK_ROOT = Path("/content/betatour_v15r2_swap_work")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
REPO = WORK_ROOT / "repo"
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "--quiet", "--depth=1", "--filter=blob:none", "--no-checkout",
                    "https://github.com/kurbankotan/betatour-tsp-lkh-candidates.git", str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "sparse-checkout", "init", "--cone"], check=True)
subprocess.run(["git", "-C", str(REPO), "sparse-checkout", "set", "src", "checkpoints"], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "--quiet", "--depth=1", "origin", REPO_COMMIT], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", "--detach", REPO_COMMIT], check=True)
assert subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip() == REPO_COMMIT
assert hashlib.sha256((REPO / "checkpoints/betatour_v2.pt").read_bytes()).hexdigest() == CHECKPOINT_SHA256
sys.path.insert(0, str(REPO / "src"))
import tsplab as T
import tsp_model as M
MODEL, _ = M.load_model(str(REPO / "checkpoints/betatour_v2.pt"), DEVICE)
assert MODEL.cfg.get("use_ensemble") is True

from google.colab import drive
drive.mount("/content/drive")
OUTPUT_ROOT = Path("/content/drive/MyDrive/BetaTour") / EXPERIMENT_NAME
for sub in ("instances", "code", "summary", "errors", "archives"):
    (OUTPUT_ROOT / sub).mkdir(parents=True, exist_ok=True)
HARDWARE_PROFILE = {"device": DEVICE.type, "gpu": GPU_NAME,
                    "vcpus": CPU_COUNT, "cpu_model": cpu_model_name()}
HARDWARE = {**HARDWARE_PROFILE,
            "python": platform.python_version(), "torch": torch.__version__,
            "numpy": np.__version__, "scipy": scipy.__version__, "pandas": pd.__version__,
            "cuda": torch.version.cuda}
if CPU_COUNT < 6:
    print("UYARI: Altı LKH süreci", CPU_COUNT, "vCPU paylaşacak; pilot yavaşlayabilir.")
print("Hazır:", HARDWARE, "kol bütçesi:", PER_ARM_BUDGET_S,
      "Drive:", OUTPUT_ROOT)


In [ ]:
BUNDLED_FILES = {'v15_live_swap/coordinator.py': '...', 'v15_live_swap/BetaTourSwapHook.h': '...', 'v15_live_swap/BetaTourVisit.h': 'LyogU2hhcmVkLCBsb3ctb3ZlcmhlYWQgY291bnRlcnMgZm9yIGFjdGlvbi1lZGdlIGVudW1lcmF0aW9uIGluIExLIGxvb3BzLiAqLwojaWZuZGVmIEJFVEFUT1VSX1ZJU0lUX0gKI2RlZmluZSBCRVRBVE9VUl9WSVNJVF9ICgpleHRlcm4gaW50ICpCVF9BY3Rpb25UbzsKZXh0ZXJuIHVuc2lnbmVkIGxvbmcgbG9uZyAqQlRfQWN0aW9uVmlzaXRzOwpleHRlcm4gdW5zaWduZWQgbG9uZyBsb25nICpCVF9BY3Rpb25GaXJzdEdhdGVQYXNzZXM7CmV4dGVybiB1bnNpZ25lZCBsb25nIGxvbmcgKkJUX0FjdGlvblBvc2l0aXZlRzE7CmV4dGVybiB1bnNpZ25lZCBsb25nIGxvbmcgKkJUX0FjdGlvbkFjY2VwdGVkRW50cmllczsKZXh0ZXJuIHVuc2lnbmVkIGxvbmcgbG9uZyBCVF9Ub3RhbENhbmRpZGF0ZUVudW1lcmF0aW9uczsKZXh0ZXJuIHVuc2lnbmVkIGxvbmcgbG9uZyBCVF9Ub3RhbEFjdGlvbkVudW1lcmF0aW9uczsKZXh0ZXJuIHVuc2lnbmVkIGxvbmcgbG9uZyBCVF9Ub3RhbFRvcExldmVsRW51bWVyYXRpb25zOwpleHRlcm4gdW5zaWduZWQgbG9uZyBsb25nIEJUX1RvdGFsVG9wTGV2ZWxBY3Rpb25FbnVtZXJhdGlvbnM7CmV4dGVybiB1bnNpZ25lZCBsb25nIGxvbmcgQlRfVG90YWxBY3Rpb25GaXJzdEdhdGVQYXNzZXM7CmV4dGVybiB1bnNpZ25lZCBsb25nIGxvbmcgQlRfVG90YWxBY3Rpb25Qb3NpdGl2ZUcxOwpleHRlcm4gdW5zaWduZWQgbG9uZyBsb25nIEJUX1RvdGFsQWN0aW9uQWNjZXB0ZWRFbnRyaWVzOwp2b2lkIEJUX0ZsdXNoVGVsZW1ldHJ5UGVyaW9kaWModm9pZCk7CgovKiBDb3VudHMgYSBkaXJlY3RlZCBjYW5kaWRhdGUgd2hlbiBhIHBhdGNoZWQgc2VhcmNoIGxvb3AgZW51bWVyYXRlcyBpdC4KICogRWxpZ2liaWxpdHkgYW5kIGdhaW4gZmlsdGVycyBtYXkgc3RpbGwgcmVqZWN0IGl0IGxhdGVyLiAqLwpzdGF0aWMgaW5saW5lIHZvaWQgQlRfTm90ZUNhbmRpZGF0ZShOb2RlICpGcm9tLCBOb2RlICpUbykKewogICAgaWYgKCFCVF9BY3Rpb25UbyB8fCAhVG8pCiAgICAgICAgcmV0dXJuOwogICAgQlRfVG90YWxDYW5kaWRhdGVFbnVtZXJhdGlvbnMrKzsKICAgIGlmIChCVF9BY3Rpb25Ub1tGcm9tLT5JZF0gPT0gVG8tPklkKSB7CiAgICAgICAgQlRfVG90YWxBY3Rpb25FbnVtZXJhdGlvbnMrKzsKICAgICAgICBCVF9BY3Rpb25WaXNpdHNbRnJvbS0+SWRdKys7CiAgICB9CiAgICBpZiAoKEJUX1RvdGFsQ2FuZGlkYXRlRW51bWVyYXRpb25zICYgNjU1MzVVTEwpID09IDApCiAgICAgICAgQlRfRmx1c2hUZWxlbWV0cnlQZXJpb2RpYygpOwp9CgovKiBUaGlzIGlzIHRoZSBmaXJzdCB0Mi0+dDMgY2hvaWNlIGluIEJlc3QyLzMvNC81L0svU3BlY2lhbE9wdE1vdmUgb25seS4gKi8Kc3RhdGljIGlubGluZSB2b2lkIEJUX05vdGVUb3BMZXZlbEVudW1lcmF0ZWQoTm9kZSAqRnJvbSwgTm9kZSAqVG8pCnsKICAgIGlmICghQlRfQWN0aW9uVG8pCiAgICAgICAgcmV0dXJuOwogICAgQlRfVG90YWxUb3BMZXZlbEVudW1lcmF0aW9ucysrOwogICAgaWYgKEJUX0FjdGlvblRvW0Zyb20tPklkXSA9PSBUby0+SWQpCiAgICAgICAgQlRfVG90YWxUb3BMZXZlbEFjdGlvbkVudW1lcmF0aW9ucysrOwp9CgovKiBDYWxsZWQgYWZ0ZXIgdGhhdCBsb29wJ3MgZmlyc3QgYWRqYWNlbmN5L0cxIGdhdGUuIFRoaXMgaXMgbm90IHRoZSBmdWxsCiAqIG11bHRpLXN0ZXAgZ2FpbiBmaWx0ZXIsIGFuZCBHMSA+IDAgaXMgdHJhY2tlZCBzZXBhcmF0ZWx5LiAqLwpzdGF0aWMgaW5saW5lIHZvaWQgQlRfTm90ZVRvcExldmVsUGFzcyhOb2RlICpGcm9tLCBOb2RlICpUbywgR2FpblR5cGUgRzEpCnsKICAgIGlmICghQlRfQWN0aW9uVG8gfHwgQlRfQWN0aW9uVG9bRnJvbS0+SWRdICE9IFRvLT5JZCkKICAgICAgICByZXR1cm47CiAgICBCVF9BY3Rpb25GaXJzdEdhdGVQYXNzZXNbRnJvbS0+SWRdKys7CiAgICBCVF9Ub3RhbEFjdGlvbkZpcnN0R2F0ZVBhc3NlcysrOwogICAgaWYgKEcxID4gMCkgewogICAgICAgIEJUX0FjdGlvblBvc2l0aXZlRzFbRnJvbS0+SWRdKys7CiAgICAgICAgQlRfVG90YWxBY3Rpb25Qb3NpdGl2ZUcxKys7CiAgICB9Cn0KCi8qIFN0b3JlVG91ciBjYWxscyB0aGlzIGJlZm9yZSByZXBsYWNpbmcgT2xkUHJlZC9PbGRTdWMgb24gYWZmZWN0ZWQgbm9kZXMuCiAqIEl0IGNvdW50cyBhbiBhY3Rpb24gZWRnZSBuZXdseSBwcmVzZW50IGluIGEgbG9jYWxseSBhY2NlcHRlZCBMSyB0b3VyLiAqLwpzdGF0aWMgaW5saW5lIHZvaWQgQlRfTm90ZUFjY2VwdGVkRWRnZShOb2RlICpGcm9tKQp7CiAgICBpbnQgVG87CiAgICBpZiAoIUJUX0FjdGlvblRvIHx8ICEoVG8gPSBCVF9BY3Rpb25Ub1tGcm9tLT5JZF0pKQogICAgICAgIHJldHVybjsKICAgIGlmICgoRnJvbS0+UHJlZC0+SWQgPT0gVG8gfHwgRnJvbS0+U3VjLT5JZCA9PSBUbykgJiYKICAgICAgICBGcm9tLT5PbGRQcmVkLT5JZCAhPSBUbyAmJiBGcm9tLT5PbGRTdWMtPklkICE9IFRvKSB7CiAgICAgICAgQlRfQWN0aW9uQWNjZXB0ZWRFbnRyaWVzW0Zyb20tPklkXSsrOwogICAgICAgIEJUX1RvdGFsQWN0aW9uQWNjZXB0ZWRFbnRyaWVzKys7CiAgICB9Cn0KCiNlbmRpZgo=', 'v15_live_swap/build_search_hook.py': 'IiIiQnVpbGQgdGhlIHYxNSBpbi1wbGFjZSBzd2FwIGhvb2sgZnJvbSBwcmlzdGluZSBMS0gtMy4wLjE0LgoKT25seSBvdXIgdHdvIGhlYWRlcnMgYW5kIGV4YWN0LXNvdXJjZSBwYXRjaGVzIGFyZSBzaGlwcGVkLiBVcHN0cmVhbSBMS0ggaXMKZG93bmxvYWRlZCBzZXBhcmF0ZWx5IGJ5IHRoZSBub3RlYm9vayBhbmQgdmVyaWZpZWQgYmVmb3JlIG1vZGlmaWNhdGlvbi4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KaW1wb3J0IHJlCmltcG9ydCBzaHV0aWwKaW1wb3J0IHN1YnByb2Nlc3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgoKVVBTVFJFQU1fU0hBMjU2ID0gewogICAgIkZpbmRUb3VyLmMiOiAiNTY3ZTcyYWNjOWI3MzkyNDg4ZGQxNThiZTRkZjAyNjc1NjA0MjQyMDRhMGU4OGQyZDliNjJmOTIzYTdjMGVlZSIsCiAgICAiQmVzdDJPcHRNb3ZlLmMiOiAiMTc1YWRlMDA4YWE4MjVkYjg5NmEzZmQxMjFlMTQ2OTFiZWM5ODNhNGI1MzljYjY5YTlkNjA4MmZmYTExN2VhZCIsCiAgICAiQmVzdDNPcHRNb3ZlLmMiOiAiMzVhYmZlOWMyZTY2ZmIwOGUzYzE1Mjk4NTU2Y2Y1MzgwMTA1OGM1M2U2NTUxYjA1YzA1YjQ4NDgxZmMwMzM1ZiIsCiAgICAiQmVzdDRPcHRNb3ZlLmMiOiAiNzA4YWM1NzU2NmZiYmQyMzU4ZTIzMDM2Yjk4NzBlNWM3N2YzZWZkNjAwYjJmMjFiODBlZWQzY2I5ODExMWJmOSIsCiAgICAiQmVzdDVPcHRNb3ZlLmMiOiAiMDlhN2I4NWQyMDBhM2E2MWYxMTA2YjQzNTMxZWE2NjM2YzVhYWQzOTdiZjllZTg2NGJiMzY5MmZjZTY3MGJjYiIsCiAgICAiQmVzdEtPcHRNb3ZlLmMiOiAiNjA4OGRiZDNlN2EzNmY0MmEyNGRmMTdlZGQ2YWQyZWU3N2I0YjIyMDdiZWY1NDU1YjliOGZjZGMzOGIzMmM3NiIsCiAgICAiQmVzdFNwZWNpYWxPcHRNb3ZlLmMiOiAiZWZiZGZiNWFlOWExMmI4MTZlMzczNDkwMTk2OWU1OWYzMWJmMDE5ZDA3MTNiMjNjOGJkMTllNGFhZjA2ZDMwMSIsCiAgICAiR2FpbjIzLmMiOiAiODUzZDg4MWI5OWM2MDM1ZTBjMDhlZGI1OGZkNWQ0ZTQ0MzdmODhjMTU0MTVkOGQxODJkYjE5NDI3ZWQ5ZDNhYiIsCiAgICAiQnJpZGdlR2Fpbi5jIjogImQ2MjVlNGQ2M2MxOTIwMzdmYzU0MGY3MjQ5ZDNlZDYyYTVmOGY3ZTJlNDRmMTA4ZDVhYjcwYzliMzE0YmI5NjMiLAogICAgIlBhdGNoQ3ljbGVzLmMiOiAiMjUzZTQ3YjFiYjY0YzExZDc2ZTM3MjQxMmI4MzM5NDExZTRkMDc5MjQxMWJlOTM3ZTQ2NjY3ODk1YmY2MGFiYyIsCiAgICAiU3RvcmVUb3VyLmMiOiAiNjgzMjU1MzYwYjhkOWQ3MWYzOTA0ODk4MDUyY2E0N2FlMWJlZjVmZWE3N2MyMzExYTIwZDE2OTY3MTE2NmY3NCIsCn0KCk1PVkVfTE9PUF9DT1VOVFMgPSB7CiAgICAiQmVzdDJPcHRNb3ZlLmMiOiAxLCAiQmVzdDNPcHRNb3ZlLmMiOiAyLAogICAgIkJlc3Q0T3B0TW92ZS5jIjogMywgIkJlc3Q1T3B0TW92ZS5jIjogNCwKICAgICJCZXN0S09wdE1vdmUuYyI6IDEsICJCZXN0U3BlY2lhbE9wdE1vdmUuYyI6IDIsCiAgICAiR2FpbjIzLmMiOiAzLCAiQnJpZGdlR2Fpbi5jIjogMywgIlBhdGNoQ3ljbGVzLmMiOiAzLAp9ClRPUF9MRVZFTF9GSUxFUyA9IHsKICAgICJCZXN0Mk9wdE1vdmUuYyIsICJCZXN0M09wdE1vdmUuYyIsICJCZXN0NE9wdE1vdmUuYyIsCiAgICAiQmVzdDVPcHRNb3ZlLmMiLCAiQmVzdEtPcHRNb3ZlLmMiLCAiQmVzdFNwZWNpYWxPcHRNb3ZlLmMiLAp9CgpfQ0FORElEQVRFX0xPT1AgPSByZS5jb21waWxlKAogICAgciIoP20pXihbIFx0XSopZm9yXHMqXChccyooTlx3Kylccyo9XHMqKFx3KyktPkNhbmRpZGF0ZVNldFxzKjsiCiAgICByIlxzKlwoXHMqKFx3Kylccyo9XHMqXDItPlRvXHMqXClccyo7XHMqXDJcK1wrXHMqXClccypceyIKKQoKCmRlZiBfcmVwbGFjZV9vbmNlKHNvdXJjZTogc3RyLCBvbGQ6IHN0ciwgbmV3OiBzdHIpIC0+IHN0cjoKICAgIGNvdW50ID0gc291cmNlLmNvdW50KG9sZCkKICAgIGlmIGNvdW50ICE9IDE6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkV4cGVjdGVkIG9uZSBzb3VyY2UgYW5jaG9yLCBmb3VuZCB7Y291bnR9OiB7b2xkWzo3MF0hcn0iKQogICAgcmV0dXJuIHNvdXJjZS5yZXBsYWNlKG9sZCwgbmV3LCAxKQoKCmRlZiBwYXRjaF9maW5kdG91cihzb3VyY2U6IHN0cikgLT4gc3RyOgogICAgc291cmNlID0gX3JlcGxhY2Vfb25jZShzb3VyY2UsICcjaW5jbHVkZSAiTEtILmgiXG4nLAogICAgICAgICAgICAgICAgICAgICAgICAgICAnI2luY2x1ZGUgIkxLSC5oIlxuI2luY2x1ZGUgIkJldGFUb3VyU3dhcEhvb2suaCJcbicpCiAgICBzb3VyY2UgPSBfcmVwbGFjZV9vbmNlKHNvdXJjZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIiAgICBkb3VibGUgRW50cnlUaW1lID0gR2V0VGltZSgpO1xuXG4gICAgdCA9IEZpcnN0Tm9kZTsiLAogICAgICAgICAgICAgICAgICAgICAgICAgICAiICAgIGRvdWJsZSBFbnRyeVRpbWUgPSBHZXRUaW1lKCk7XG5cbiIKICAgICAgICAgICAgICAgICAgICAgICAgICAgIiAgICBCVF9FeHBvcnROYXRpdmVDYW5kaWRhdGVzKCk7XG4gICAgdCA9IEZpcnN0Tm9kZTsiKQogICAgc291cmNlID0gX3JlcGxhY2Vfb25jZShzb3VyY2UsICIgICAgICAgICAgICBSZWNvcmRCZXR0ZXJUb3VyKCk7XG4iLAogICAgICAgICAgICAgICAgICAgICAgICAgICAiICAgICAgICAgICAgUmVjb3JkQmV0dGVyVG91cigpO1xuICAgICAgICAgICAgQlRfUmVjb3JkQmVzdCgpO1xuIikKICAgIHNvdXJjZSA9IF9yZXBsYWNlX29uY2Uoc291cmNlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAiICAgICAgICB9XG4gICAgfVxuICAgIGlmIChCYWNrYm9uZVRyaWFscyA+IDAgJiYgQmFja2JvbmVUcmlhbHMgPCBNYXhUcmlhbHMpIHsiLAogICAgICAgICAgICAgICAgICAgICAgICAgICAiICAgICAgICB9XG4gICAgICAgIEJUX0FmdGVyRmlyc3RUcmlhbCgpO1xuIgogICAgICAgICAgICAgICAgICAgICAgICAgICAiICAgIH1cbiAgICBpZiAoQmFja2JvbmVUcmlhbHMgPiAwICYmIEJhY2tib25lVHJpYWxzIDwgTWF4VHJpYWxzKSB7IikKICAgIHJldHVybiBzb3VyY2UKCgpkZWYgcGF0Y2hfbW92ZV9maWxlKHNvdXJjZTogc3RyLCBmaWxlbmFtZTogc3RyKSAtPiBzdHI6CiAgICBzb3VyY2UgPSBfcmVwbGFjZV9vbmNlKHNvdXJjZSwgJyNpbmNsdWRlICJMS0guaCJcbicsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICcjaW5jbHVkZSAiTEtILmgiXG4jaW5jbHVkZSAiQmV0YVRvdXJWaXNpdC5oIlxuJykKCiAgICBkZWYgaW5zZXJ0X3Zpc2l0KG1hdGNoOiByZS5NYXRjaFtzdHJdKSAtPiBzdHI6CiAgICAgICAgaW5kZW50LCBfY2FuZGlkYXRlLCBmcm9tX25hbWUsIHRvX25hbWUgPSBtYXRjaC5ncm91cHMoKQogICAgICAgIHJldHVybiBtYXRjaC5ncm91cCgwKSArIGYiXG57aW5kZW50fSAgICBCVF9Ob3RlQ2FuZGlkYXRlKHtmcm9tX25hbWV9LCB7dG9fbmFtZX0pOyIKCiAgICBzb3VyY2UsIGNvdW50ID0gX0NBTkRJREFURV9MT09QLnN1Ym4oaW5zZXJ0X3Zpc2l0LCBzb3VyY2UpCiAgICBpZiBjb3VudCAhPSBNT1ZFX0xPT1BfQ09VTlRTW2ZpbGVuYW1lXToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiVW5leHBlY3RlZCBjYW5kaWRhdGUgbG9vcHMgaW4ge2ZpbGVuYW1lfToge2NvdW50fSIpCiAgICBpZiBmaWxlbmFtZSBpbiBUT1BfTEVWRUxfRklMRVM6CiAgICAgICAgb2xkID0gIkJUX05vdGVDYW5kaWRhdGUodDIsIHQzKTsiCiAgICAgICAgc291cmNlID0gX3JlcGxhY2Vfb25jZShzb3VyY2UsIG9sZCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG9sZCArICJcbiAgICAgICAgQlRfTm90ZVRvcExldmVsRW51bWVyYXRlZCh0MiwgdDMpOyIpCiAgICAgICAgZmlyc3QgPSBzb3VyY2UuaW5kZXgoIkJUX05vdGVUb3BMZXZlbEVudW1lcmF0ZWQodDIsIHQzKTsiKQogICAgICAgIGdhdGUgPSAiICAgICAgICAgICAgY29udGludWU7XG4iCiAgICAgICAgZW5kID0gc291cmNlLmZpbmQoZ2F0ZSwgZmlyc3QpCiAgICAgICAgaWYgZW5kIDwgMDoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkNhbm5vdCBmaW5kIGZpcnN0LXN0YWdlIGdhaW4gZ2F0ZSBpbiB7ZmlsZW5hbWV9IikKICAgICAgICBlbmQgKz0gbGVuKGdhdGUpCiAgICAgICAgc291cmNlID0gc291cmNlWzplbmRdICsgIiAgICAgICAgQlRfTm90ZVRvcExldmVsUGFzcyh0MiwgdDMsIEcxKTtcbiIgKyBzb3VyY2VbZW5kOl0KICAgIHJldHVybiBzb3VyY2UKCgpkZWYgcGF0Y2hfc3RvcmV0b3VyKHNvdXJjZTogc3RyKSAtPiBzdHI6CiAgICBzb3VyY2UgPSBfcmVwbGFjZV9vbmNlKHNvdXJjZSwgJyNpbmNsdWRlICJMS0guaCJcbicsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICcjaW5jbHVkZSAiTEtILmgiXG4jaW5jbHVkZSAiQmV0YVRvdXJWaXNpdC5oIlxuJykKICAgIHNvdXJjZSA9IF9yZXBsYWNlX29uY2Uoc291cmNlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAiICAgICAgICAgICAgQWN0aXZhdGUodCk7XG4gICAgICAgICAgICB0LT5PbGRQcmVkID0gdC0+UHJlZDsiLAogICAgICAgICAgICAgICAgICAgICAgICAgICAiICAgICAgICAgICAgQWN0aXZhdGUodCk7XG4iCiAgICAgICAgICAgICAgICAgICAgICAgICAgICIgICAgICAgICAgICBCVF9Ob3RlQWNjZXB0ZWRFZGdlKHQpO1xuIgogICAgICAgICAgICAgICAgICAgICAgICAgICAiICAgICAgICAgICAgdC0+T2xkUHJlZCA9IHQtPlByZWQ7IikKICAgIHJldHVybiBzb3VyY2UKCgpkZWYgX2NvbXBhY3QodGV4dDogc3RyKSAtPiBzdHI6CiAgICByZXR1cm4gcmUuc3ViKHIiXHMrIiwgIiIsIHRleHQpCgoKZGVmIGluc3BlY3RfY2FuZGlkYXRlX3Jlb3JkZXJpbmcoc3JjOiBQYXRoKSAtPiBkaWN0OgogICAgIiIiUmVjb3JkLCB3aXRob3V0IHBhdGNoaW5nLCBob3cgdXBzdHJlYW0gTEtIIHJlb3JkZXJzIGNhbmRpZGF0ZSByb3dzLgoKICAgIFRoZSB2MTUgaG9vayBzd2FwcyBhIGNhbmRpZGF0ZSBhdCB0aGUgYmFycmllciBhbmQgZGVsaWJlcmF0ZWx5IGRvZXMgbm90CiAgICBlbmZvcmNlIHJhbmsgb3IgbWVtYmVyc2hpcCBhZnRlciBMS0ggcmVzdW1lcy4gVGhpcyBkb2N1bWVudHMgdGhlIG5hdGl2ZQogICAgcmVvcmRlcmluZyBiZWhhdmlvcjsgaXQgbmV2ZXIgZmFpbHMgdGhlIGJ1aWxkLgogICAgIiIiCiAgICByZXBvcnQ6IGRpY3QgPSB7fQogICAgdGV4dHMgPSB7fQogICAgZm9yIG5hbWUgaW4gKCJGaW5kVG91ci5jIiwgIkFkanVzdENhbmRpZGF0ZVNldC5jIiwgIlJlc2V0Q2FuZGlkYXRlU2V0LmMiKToKICAgICAgICBwYXRoID0gc3JjIC8gbmFtZQogICAgICAgIGlmIHBhdGguZXhpc3RzKCk6CiAgICAgICAgICAgIGRhdGEgPSBwYXRoLnJlYWRfYnl0ZXMoKQogICAgICAgICAgICB0ZXh0c1tuYW1lXSA9IF9jb21wYWN0KGRhdGEuZGVjb2RlKCJ1dGYtOCIsIGVycm9ycz0icmVwbGFjZSIpKQogICAgICAgICAgICByZXBvcnRbbmFtZS5yZXBsYWNlKCIuYyIsICJfc2hhMjU2IildID0gaGFzaGxpYi5zaGEyNTYoZGF0YSkuaGV4ZGlnZXN0KCkKICAgICAgICBlbHNlOgogICAgICAgICAgICByZXBvcnRbbmFtZS5yZXBsYWNlKCIuYyIsICJfc2hhMjU2IildID0gTm9uZQogICAgZmluZHRvdXIgPSB0ZXh0cy5nZXQoIkZpbmRUb3VyLmMiLCAiIikKICAgIGxvb3AgPSBmaW5kdG91ci5zcGxpdCgiZm9yKFRyaWFsPTE7VHJpYWw8PU1heFRyaWFscztUcmlhbCsrKSIsIDEpCiAgICByZXBvcnRbImZpbmR0b3VyX3RyaWFsX2xvb3BfY2FsbHNfYWRqdXN0X2NhbmRpZGF0ZV9zZXQiXSA9ICgKICAgICAgICBsZW4obG9vcCkgPT0gMiBhbmQgIkFkanVzdENhbmRpZGF0ZVNldCgpOyIgaW4gbG9vcFsxXS5zcGxpdCgiUmVzZXRDYW5kaWRhdGVTZXQoKTsiLCAxKVswXSkKICAgIGFkanVzdCA9IHRleHRzLmdldCgiQWRqdXN0Q2FuZGlkYXRlU2V0LmMiLCAiIikKICAgIHJlcG9ydFsiYWRqdXN0X21vdmVzX2VkZ2VzX29mX3R3b19iZXN0X3RvdXJzX3RvX2Zyb250Il0gPSAoCiAgICAgICAgIkluQmVzdFRvdXIoRnJvbSxUbykmJkluTmV4dEJlc3RUb3VyKEZyb20sVG8pIiBpbiBhZGp1c3QpCiAgICByZXBvcnRbImFkanVzdF9zb3J0c19yb3dzX2J5X2FscGhhX3RoZW5fY29zdCJdID0gKAogICAgICAgICJUZW1wLkFscGhhPE5OLT5BbHBoYSIgaW4gYWRqdXN0KQogICAgcmVwb3J0WyJyZXNldF9zb3J0c19yb3dzX2J5X2FscGhhX3RoZW5fY29zdCJdID0gKAogICAgICAgICJUZW1wLkFscGhhPE5OLT5BbHBoYSIgaW4gdGV4dHMuZ2V0KCJSZXNldENhbmRpZGF0ZVNldC5jIiwgIiIpKQogICAgcmV0dXJuIHJlcG9ydAoKCmRlZiBidWlsZChzb3VyY2VfZGlyOiBQYXRoLCBvdXRwdXRfZGlyOiBQYXRoLCBqb2JzOiBpbnQgPSA0KSAtPiBQYXRoOgogICAgc291cmNlX2Rpciwgb3V0cHV0X2RpciA9IHNvdXJjZV9kaXIucmVzb2x2ZSgpLCBvdXRwdXRfZGlyLnJlc29sdmUoKQogICAgc3JjID0gc291cmNlX2RpciAvICJTUkMiCiAgICBpZiBub3QgKHNvdXJjZV9kaXIgLyAiTWFrZWZpbGUiKS5leGlzdHMoKToKICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmIkxLSCBzb3VyY2UgdHJlZSBtaXNzaW5nOiB7c291cmNlX2Rpcn0iKQogICAgb3JpZ2luYWxzID0ge30KICAgIGZvciBmaWxlbmFtZSwgZXhwZWN0ZWQgaW4gVVBTVFJFQU1fU0hBMjU2Lml0ZW1zKCk6CiAgICAgICAgZGF0YSA9IChzcmMgLyBmaWxlbmFtZSkucmVhZF9ieXRlcygpCiAgICAgICAgZGlnZXN0ID0gaGFzaGxpYi5zaGEyNTYoZGF0YSkuaGV4ZGlnZXN0KCkKICAgICAgICBpZiBkaWdlc3QgIT0gZXhwZWN0ZWQ6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJVbmV4cGVjdGVkIHVwc3RyZWFtIFNIQTI1NiBmb3Ige2ZpbGVuYW1lfToge2RpZ2VzdH0iKQogICAgICAgIG9yaWdpbmFsc1tmaWxlbmFtZV0gPSBkYXRhLmRlY29kZSgidXRmLTgiKQogICAgaWYgb3V0cHV0X2Rpci5leGlzdHMoKToKICAgICAgICByYWlzZSBGaWxlRXhpc3RzRXJyb3IoZiJSZWZ1c2luZyB0byBvdmVyd3JpdGUge291dHB1dF9kaXJ9IikKICAgIHNodXRpbC5jb3B5dHJlZShzb3VyY2VfZGlyLCBvdXRwdXRfZGlyLAogICAgICAgICAgICAgICAgICAgIGlnbm9yZT1zaHV0aWwuaWdub3JlX3BhdHRlcm5zKCJPQkoiLCAiTEtIIiwgIioubyIpKQogICAgKG91dHB1dF9kaXIgLyAiU1JDIiAvICJPQkoiKS5ta2RpcihleGlzdF9vaz1UcnVlKQogICAgKG91dHB1dF9kaXIgLyAiU1JDIiAvICJGaW5kVG91ci5jIikud3JpdGVfdGV4dCgKICAgICAgICBwYXRjaF9maW5kdG91cihvcmlnaW5hbHNbIkZpbmRUb3VyLmMiXSkpCiAgICBmb3IgZmlsZW5hbWUgaW4gTU9WRV9MT09QX0NPVU5UUzoKICAgICAgICAob3V0cHV0X2RpciAvICJTUkMiIC8gZmlsZW5hbWUpLndyaXRlX3RleHQoCiAgICAgICAgICAgIHBhdGNoX21vdmVfZmlsZShvcmlnaW5hbHNbZmlsZW5hbWVdLCBmaWxlbmFtZSkpCiAgICAob3V0cHV0X2RpciAvICJTUkMiIC8gIlN0b3JlVG91ci5jIikud3JpdGVfdGV4dCgKICAgICAgICBwYXRjaF9zdG9yZXRvdXIob3JpZ2luYWxzWyJTdG9yZVRvdXIuYyJdKSkKICAgIGZvciBoZWFkZXIgaW4gKCJCZXRhVG91clN3YXBIb29rLmgiLCAiQmV0YVRvdXJWaXNpdC5oIik6CiAgICAgICAgc2h1dGlsLmNvcHkyKFBhdGgoX19maWxlX18pLndpdGhfbmFtZShoZWFkZXIpLCBvdXRwdXRfZGlyIC8gIlNSQyIgLyBoZWFkZXIpCiAgICByZXN1bHQgPSBzdWJwcm9jZXNzLnJ1bihbIm1ha2UiLCAiLWoiLCBzdHIoam9icyldLCBjd2Q9b3V0cHV0X2RpciwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIHN0ZG91dD1zdWJwcm9jZXNzLlBJUEUsIHN0ZGVycj1zdWJwcm9jZXNzLlNURE9VVCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIHRleHQ9VHJ1ZSwgY2hlY2s9RmFsc2UpCiAgICBpZiByZXN1bHQucmV0dXJuY29kZToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJMS0ggaW5zdHJ1bWVudGVkIGJ1aWxkIGZhaWxlZDpcbntyZXN1bHQuc3Rkb3V0Wy04MDAwOl19IikKICAgIGJpbmFyeSA9IG91dHB1dF9kaXIgLyAiTEtIIgogICAgaWYgbm90IGJpbmFyeS5leGlzdHMoKToKICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmIk1pc3NpbmcgYnVpbHQgYmluYXJ5OiB7YmluYXJ5fSIpCiAgICBtYW5pZmVzdCA9IHsidXBzdHJlYW1fc2hhMjU2IjogVVBTVFJFQU1fU0hBMjU2LAogICAgICAgICAgICAgICAgInNlYXJjaF9ob29rX3NoYTI1NiI6IGhhc2hsaWIuc2hhMjU2KAogICAgICAgICAgICAgICAgICAgIFBhdGgoX19maWxlX18pLndpdGhfbmFtZSgiQmV0YVRvdXJTd2FwSG9vay5oIikucmVhZF9ieXRlcygpKS5oZXhkaWdlc3QoKSwKICAgICAgICAgICAgICAgICJ2aXNpdF9oZWFkZXJfc2hhMjU2IjogaGFzaGxpYi5zaGEyNTYoCiAgICAgICAgICAgICAgICAgICAgUGF0aChfX2ZpbGVfXykud2l0aF9uYW1lKCJCZXRhVG91clZpc2l0LmgiKS5yZWFkX2J5dGVzKCkpLmhleGRpZ2VzdCgpLAogICAgICAgICAgICAgICAgImJpbmFyeV9zaGEyNTYiOiBoYXNobGliLnNoYTI1NihiaW5hcnkucmVhZF9ieXRlcygpKS5oZXhkaWdlc3QoKSwKICAgICAgICAgICAgICAgICJpbnN0cnVtZW50ZWRfbW92ZV9sb29wcyI6IHN1bShNT1ZFX0xPT1BfQ09VTlRTLnZhbHVlcygpKSwKICAgICAgICAgICAgICAgICJzd2FwX2F0X2JhcnJpZXIiOiBUcnVlLAogICAgICAgICAgICAgICAgImFjdGlvbl9ydWxlIjogImluLXBsYWNlIGRpcmVjdGVkIGNhbmRpZGF0ZSByZXBsYWNlbWVudCBvbmNlIGFmdGVyIGZpcnN0IHRyaWFsOyBubyBzbG90IHJlc3RvcmF0aW9uIiwKICAgICAgICAgICAgICAgICJ1cHN0cmVhbV9jYW5kaWRhdGVfcmVvcmRlcmluZyI6IGluc3BlY3RfY2FuZGlkYXRlX3Jlb3JkZXJpbmcoc3JjKX0KICAgIChvdXRwdXRfZGlyIC8gImJldGF0b3VyX2J1aWxkLmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMobWFuaWZlc3QsIGluZGVudD0yKSArICJcbiIpCiAgICByZXR1cm4gYmluYXJ5CgoKZGVmIG1haW4oKSAtPiBOb25lOgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249X19kb2NfXykKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tc291cmNlLWRpciIsIHR5cGU9UGF0aCwgcmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tb3V0cHV0LWRpciIsIHR5cGU9UGF0aCwgcmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tam9icyIsIHR5cGU9aW50LCBkZWZhdWx0PTQpCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgcHJpbnQoYnVpbGQoYXJncy5zb3VyY2VfZGlyLCBhcmdzLm91dHB1dF9kaXIsIGFyZ3Muam9icykpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=', 'v15_live_swap/test_search_hook.py': 'IiIiUmVhbCBMS0gtMy4wLjE0IHNtb2tlIG9mIHRoZSB2MTUgZmlyc3QtdHJpYWwgc3dhcCBob29rLgoKVXNhZ2U6IHB5dGhvbiB0ZXN0X3NlYXJjaF9ob29rLnB5IC0tc291cmNlLWRpciBMS0gtMy4wLjE0IC0tYmluYXJ5IFBBVENIRUQvTEtICiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBvcwppbXBvcnQgcmFuZG9tCmltcG9ydCBzdWJwcm9jZXNzCmltcG9ydCB0ZW1wZmlsZQppbXBvcnQgdGltZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmZyb20gYnVpbGRfc2VhcmNoX2hvb2sgaW1wb3J0ICgKICAgIE1PVkVfTE9PUF9DT1VOVFMsIFVQU1RSRUFNX1NIQTI1NiwgcGF0Y2hfZmluZHRvdXIsIHBhdGNoX21vdmVfZmlsZSwKICAgIHBhdGNoX3N0b3JldG91ciwKKQoKCmRlZiBzb3VyY2VfY2hlY2tzKHNvdXJjZV9kaXI6IFBhdGgpIC0+IE5vbmU6CiAgICBzcmMgPSBzb3VyY2VfZGlyIC8gIlNSQyIKICAgIHBhdGNoZWQgPSBwYXRjaF9maW5kdG91cigoc3JjIC8gIkZpbmRUb3VyLmMiKS5yZWFkX3RleHQoKSkKICAgIGFzc2VydCAnI2luY2x1ZGUgIkJldGFUb3VyU3dhcEhvb2suaCInIGluIHBhdGNoZWQKICAgIGFzc2VydCAiQlRfQWZ0ZXJGaXJzdFRyaWFsKCkiIGluIHBhdGNoZWQKICAgIGZvciBuYW1lLCBjb3VudCBpbiBNT1ZFX0xPT1BfQ09VTlRTLml0ZW1zKCk6CiAgICAgICAgcGF0Y2hlZCA9IHBhdGNoX21vdmVfZmlsZSgoc3JjIC8gbmFtZSkucmVhZF90ZXh0KCksIG5hbWUpCiAgICAgICAgYXNzZXJ0IHBhdGNoZWQuY291bnQoIkJUX05vdGVDYW5kaWRhdGUoIikgPT0gY291bnQKICAgIHBhdGNoZWQgPSBwYXRjaF9zdG9yZXRvdXIoKHNyYyAvICJTdG9yZVRvdXIuYyIpLnJlYWRfdGV4dCgpKQogICAgYXNzZXJ0IHBhdGNoZWQuY291bnQoIkJUX05vdGVBY2NlcHRlZEVkZ2UodCkiKSA9PSAxCiAgICBhc3NlcnQgbGVuKFVQU1RSRUFNX1NIQTI1NikgPT0gMTEKCgpkZWYgX3JlYWRfdG91cihwYXRoOiBQYXRoKSAtPiBsaXN0W2ludF06CiAgICBmaWVsZHMgPSBbaW50KHgpIGZvciB4IGluIHBhdGgucmVhZF90ZXh0KCkuc3BsaXQoKV0KICAgIGFzc2VydCBsZW4oZmllbGRzKSA9PSBmaWVsZHNbMF0gKyAxCiAgICB0b3VyID0gZmllbGRzWzE6XQogICAgYXNzZXJ0IHNvcnRlZCh0b3VyKSA9PSBsaXN0KHJhbmdlKDEsIGxlbih0b3VyKSArIDEpKQogICAgcmV0dXJuIHRvdXIKCgpkZWYgX3JlYWRfcm93cyhwYXRoOiBQYXRoKSAtPiBsaXN0W2xpc3RbdHVwbGVbaW50LCBpbnRdXV06CiAgICBsaW5lcyA9IHBhdGgucmVhZF90ZXh0KCkuc3BsaXRsaW5lcygpCiAgICBuID0gaW50KGxpbmVzWzBdKQogICAgYXNzZXJ0IGxlbihsaW5lcykgPT0gbiArIDEKICAgIHJvd3MgPSBbXQogICAgZm9yIHNvdXJjZSwgbGluZSBpbiBlbnVtZXJhdGUobGluZXNbMTpdLCAxKToKICAgICAgICBmaWVsZHMgPSBbaW50KHgpIGZvciB4IGluIGxpbmUuc3BsaXQoKV0KICAgICAgICBhc3NlcnQgZmllbGRzWzBdID09IHNvdXJjZSBhbmQgbGVuKGZpZWxkcykgPT0gMiArIDIgKiBmaWVsZHNbMV0KICAgICAgICByb3dzLmFwcGVuZChbKGZpZWxkc1tpXSwgZmllbGRzW2kgKyAxXSkgZm9yIGkgaW4gcmFuZ2UoMiwgbGVuKGZpZWxkcyksIDIpXSkKICAgIHJldHVybiByb3dzCgoKZGVmIF9lZGdlcyh0b3VyOiBsaXN0W2ludF0pIC0+IHNldFt0dXBsZVtpbnQsIGludF1dOgogICAgcmV0dXJuIHt0dXBsZShzb3J0ZWQoKGEsIGIpKSkgZm9yIGEsIGIgaW4gemlwKHRvdXIsIHRvdXJbMTpdICsgdG91cls6MV0pfQoKCmRlZiBydW5fc21va2UoYmluYXJ5OiBQYXRoKSAtPiBOb25lOgogICAgd2l0aCB0ZW1wZmlsZS5UZW1wb3JhcnlEaXJlY3RvcnkocHJlZml4PSJiZXRhdG91cl92MTVfcmVhbF8iKSBhcyB0ZW1wOgogICAgICAgIHJvb3QgPSBQYXRoKHRlbXApCiAgICAgICAgaG9vayA9IHJvb3QgLyAiaG9vayIKICAgICAgICBob29rLm1rZGlyKCkKICAgICAgICBuID0gMTgwCiAgICAgICAgcm5nID0gcmFuZG9tLlJhbmRvbSgxMTcpCiAgICAgICAgY29vcmRzID0gWyhybmcucmFuZHJhbmdlKDEwMDAwKSwgcm5nLnJhbmRyYW5nZSgxMDAwMCkpIGZvciBfIGluIHJhbmdlKG4pXQogICAgICAgIHByb2JsZW0gPSByb290IC8gInNtb2tlLnRzcCIKICAgICAgICBwcm9ibGVtLndyaXRlX3RleHQoCiAgICAgICAgICAgICJOQU1FIDogc21va2VcblRZUEUgOiBUU1BcbkRJTUVOU0lPTiA6IDE4MFxuIgogICAgICAgICAgICAiRURHRV9XRUlHSFRfVFlQRSA6IEVVQ18yRFxuTk9ERV9DT09SRF9TRUNUSU9OXG4iCiAgICAgICAgICAgICsgIiIuam9pbihmIntpICsgMX0ge3h9IHt5fVxuIiBmb3IgaSwgKHgsIHkpIGluIGVudW1lcmF0ZShjb29yZHMpKQogICAgICAgICAgICArICJFT0ZcbiIpCiAgICAgICAgcGFyYW1zID0gaG9vayAvICJydW4ucGFyIgogICAgICAgIHBhcmFtcy53cml0ZV90ZXh0KAogICAgICAgICAgICBmIlBST0JMRU1fRklMRSA9IHtwcm9ibGVtfVxuQ0FORElEQVRFX1NFVF9UWVBFID0gUE9QTVVTSUNcbiIKICAgICAgICAgICAgIk1BWF9DQU5ESURBVEVTID0gNlxuU1VCR1JBRElFTlQgPSBOT1xuU0VFRCA9IDMwMVxuIgogICAgICAgICAgICAiUlVOUyA9IDFcbk1BWF9UUklBTFMgPSAxMDBcblRJTUVfTElNSVQgPSAxMDBcbiIKICAgICAgICAgICAgIlRSQUNFX0xFVkVMID0gMFxuU1RPUF9BVF9PUFRJTVVNID0gTk9cbiIpCiAgICAgICAgZW52ID0gZGljdChvcy5lbnZpcm9uLCBCRVRBVE9VUl9IT09LX0RJUj1zdHIoaG9vayksCiAgICAgICAgICAgICAgICAgICBCRVRBVE9VUl9IT09LX1RJTUVPVVRfUz0iMTIiKQogICAgICAgIHdpdGggKGhvb2sgLyAic3Rkb3V0LnR4dCIpLm9wZW4oInciKSBhcyBvdXQsIChob29rIC8gInN0ZGVyci50eHQiKS5vcGVuKCJ3IikgYXMgZXJyOgogICAgICAgICAgICBwcm9jZXNzID0gc3VicHJvY2Vzcy5Qb3Blbihbc3RyKGJpbmFyeS5yZXNvbHZlKCkpLCBzdHIocGFyYW1zKV0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGN3ZD1ob29rLCBzdGRvdXQ9b3V0LCBzdGRlcnI9ZXJyLCBlbnY9ZW52KQogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBkZWFkbGluZSA9IHRpbWUubW9ub3RvbmljKCkgKyAxMgogICAgICAgICAgICAgICAgd2hpbGUgbm90IChob29rIC8gImZpcnN0LnRvdXIiKS5leGlzdHMoKSBhbmQgdGltZS5tb25vdG9uaWMoKSA8IGRlYWRsaW5lOgogICAgICAgICAgICAgICAgICAgIGlmIHByb2Nlc3MucG9sbCgpIGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkxLSCBleGl0ZWQgYmVmb3JlIGJhcnJpZXI6ICIKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICsgKGhvb2sgLyAic3RkZXJyLnR4dCIpLnJlYWRfdGV4dCgpKQogICAgICAgICAgICAgICAgICAgIHRpbWUuc2xlZXAoMC4wMikKICAgICAgICAgICAgICAgIGFzc2VydCAoaG9vayAvICJmaXJzdC50b3VyIikuZXhpc3RzKCksICJmaXJzdC10cmlhbCBiYXJyaWVyIHRpbWVvdXQiCiAgICAgICAgICAgICAgICB0b3VyID0gX3JlYWRfdG91cihob29rIC8gImZpcnN0LnRvdXIiKQogICAgICAgICAgICAgICAgbGl2ZSA9IF9yZWFkX3Jvd3MoaG9vayAvICJsaXZlLmNhbmRpZGF0ZXMiKQogICAgICAgICAgICAgICAgcHJvdGVjdGVkID0gX2VkZ2VzKHRvdXIpCiAgICAgICAgICAgICAgICBncmFwaCA9IHt0dXBsZShzb3J0ZWQoKHUsIHYpKSkgZm9yIHUsIHJvdyBpbiBlbnVtZXJhdGUobGl2ZSwgMSkKICAgICAgICAgICAgICAgICAgICAgICAgIGZvciB2LCBfYWxwaGEgaW4gcm93fQogICAgICAgICAgICAgICAgYWN0aW9uID0gTm9uZQogICAgICAgICAgICAgICAgZm9yIHUsIHJvdyBpbiBlbnVtZXJhdGUobGl2ZSwgMSk6CiAgICAgICAgICAgICAgICAgICAgZm9yIHZpY3RpbSwgX2FscGhhIGluIHJvd1syOl06CiAgICAgICAgICAgICAgICAgICAgICAgIGlmIHR1cGxlKHNvcnRlZCgodSwgdmljdGltKSkpIGluIHByb3RlY3RlZDoKICAgICAgICAgICAgICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgICAgICAgICAgICAgIGZvciB2IGluIHJhbmdlKDEsIG4gKyAxKToKICAgICAgICAgICAgICAgICAgICAgICAgICAgIGlmIHYgIT0gdSBhbmQgdHVwbGUoc29ydGVkKCh1LCB2KSkpIG5vdCBpbiBncmFwaDoKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBhY3Rpb24gPSAodSwgdiwgdmljdGltKQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgICAgICAgICAgICAgIGlmIGFjdGlvbjoKICAgICAgICAgICAgICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgICAgICAgICAgaWYgYWN0aW9uOgogICAgICAgICAgICAgICAgICAgICAgICBicmVhawogICAgICAgICAgICAgICAgYXNzZXJ0IGFjdGlvbiBpcyBub3QgTm9uZSwgIm5vIHNhZmUgc21va2UgYWN0aW9uIgogICAgICAgICAgICAgICAgZWRnZV9saW5lcyA9IFtmInthfSB7Yn0iIGZvciBhLCBiIGluIHNvcnRlZChwcm90ZWN0ZWQpXQogICAgICAgICAgICAgICAgKGhvb2sgLyAicHJvdGVjdGVkLmVkZ2VzIikud3JpdGVfdGV4dCgKICAgICAgICAgICAgICAgICAgICBmIntufSB7bGVuKGVkZ2VfbGluZXMpfVxuIiArICJcbiIuam9pbihlZGdlX2xpbmVzKSArICJcbiIpCiAgICAgICAgICAgICAgICB1LCB2LCB2aWN0aW0gPSBhY3Rpb24KICAgICAgICAgICAgICAgIGFjdGlvbl9saW5lcyA9IFtzdHIobildICsgWwogICAgICAgICAgICAgICAgICAgIGYie3NvdXJjZX0ge3Z9IHt2aWN0aW19IDUwMCIgaWYgc291cmNlID09IHUgZWxzZQogICAgICAgICAgICAgICAgICAgIGYie3NvdXJjZX0gMCAwIDAiIGZvciBzb3VyY2UgaW4gcmFuZ2UoMSwgbiArIDEpXQogICAgICAgICAgICAgICAgdGVtcG9yYXJ5ID0gaG9vayAvICJhY3Rpb24udHh0LnRtcCIKICAgICAgICAgICAgICAgIHRlbXBvcmFyeS53cml0ZV90ZXh0KCJcbiIuam9pbihhY3Rpb25fbGluZXMpICsgIlxuIikKICAgICAgICAgICAgICAgIG9zLnJlcGxhY2UodGVtcG9yYXJ5LCBob29rIC8gImFjdGlvbi50eHQiKQogICAgICAgICAgICAgICAgd2hpbGUgbm90IChob29rIC8gInBvc3QuY2FuZGlkYXRlcyIpLmV4aXN0cygpIGFuZCB0aW1lLm1vbm90b25pYygpIDwgZGVhZGxpbmU6CiAgICAgICAgICAgICAgICAgICAgaWYgcHJvY2Vzcy5wb2xsKCkgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiTEtIIGV4aXRlZCBhdCBzd2FwOiAiCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICArIChob29rIC8gInN0ZGVyci50eHQiKS5yZWFkX3RleHQoKSkKICAgICAgICAgICAgICAgICAgICB0aW1lLnNsZWVwKDAuMDIpCiAgICAgICAgICAgICAgICBhc3NlcnQgKGhvb2sgLyAicG9zdC5jYW5kaWRhdGVzIikuZXhpc3RzKCksICJwb3N0LXN3YXAgdGltZW91dCIKICAgICAgICAgICAgICAgIHBvc3QgPSBfcmVhZF9yb3dzKGhvb2sgLyAicG9zdC5jYW5kaWRhdGVzIikKICAgICAgICAgICAgICAgIGV4cGVjdGVkID0gbGlzdChsaXZlW3UgLSAxXSkKICAgICAgICAgICAgICAgIGluZGV4ID0gbmV4dChpIGZvciBpLCAodG8sIF9hbHBoYSkgaW4gZW51bWVyYXRlKGV4cGVjdGVkKQogICAgICAgICAgICAgICAgICAgICAgICAgICAgIGlmIHRvID09IHZpY3RpbSkKICAgICAgICAgICAgICAgIGV4cGVjdGVkW2luZGV4XSA9ICh2LCA1MDApCiAgICAgICAgICAgICAgICBhc3NlcnQgcG9zdFt1IC0gMV0gPT0gZXhwZWN0ZWQKICAgICAgICAgICAgICAgIGFzc2VydCBhbGwocG9zdFtpXSA9PSBsaXZlW2ldIGZvciBpIGluIHJhbmdlKG4pIGlmIGkgIT0gdSAtIDEpCiAgICAgICAgICAgICAgICBzdGF0cyA9IGRpY3QobGluZS5zcGxpdCgpIGZvciBsaW5lIGluCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgKGhvb2sgLyAiYXBwbGllZC5zdGF0cyIpLnJlYWRfdGV4dCgpLnNwbGl0bGluZXMoKSkKICAgICAgICAgICAgICAgIGFzc2VydCBzdGF0c1sic3dhcHBlZCJdID09ICIxIiBhbmQgc3RhdHNbInJlamVjdGVkIl0gPT0gIjAiCiAgICAgICAgICAgIGZpbmFsbHk6CiAgICAgICAgICAgICAgICBpZiBwcm9jZXNzLnBvbGwoKSBpcyBOb25lOgogICAgICAgICAgICAgICAgICAgIHByb2Nlc3MudGVybWluYXRlKCkKICAgICAgICAgICAgICAgIHByb2Nlc3Mud2FpdCh0aW1lb3V0PTMpCgoKZGVmIG1haW4oKSAtPiBOb25lOgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249X19kb2NfXykKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tc291cmNlLWRpciIsIHR5cGU9UGF0aCwgcmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tYmluYXJ5IiwgdHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlKQogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKCkKICAgIHNvdXJjZV9jaGVja3MoYXJncy5zb3VyY2VfZGlyKQogICAgcnVuX3Ntb2tlKGFyZ3MuYmluYXJ5KQogICAgcHJpbnQoInYxNSByZWFsIExLSCBzd2FwIHNtb2tlIHBhc3NlZCIpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=', 'v15_live_swap/policy.py': '...', 'v15_live_swap/pilot.py': '...', 'v15_live_swap/run_cell.py': 'IiIidjE1IGxpdmUtc3dhcCBleGVjdXRpb24gY2VsbC4gQ29tcGxldGVkIGFybXMgYXJlIGltbXV0YWJsZSBhbmQgcmVzdW1hYmxlLiIiIgoKCmRlZiBhcm1fcmVjb3JkX3BhdGgoaW5zdCwgYXJtKToKICAgIHJldHVybiBPVVRQVVRfUk9PVCAvICJpbnN0YW5jZXMiIC8gaW5zdFsibmFtZSJdIC8gYXJtIC8gInJlc3VsdC5qc29uIgoKCmRlZiB2YWxpZGF0ZV9zYXZlZChpbnN0LCBhcm0sIHJlY29yZCk6CiAgICBhc3NlcnQgcmVjb3JkWyJjb21wbGV0ZSJdIGFuZCByZWNvcmRbImFybSJdID09IGFybSBhbmQgcmVjb3JkWyJpbnN0YW5jZSJdID09IGluc3RbIm5hbWUiXQogICAgYXNzZXJ0IHJlY29yZFsieHlfc2hhMjU2Il0gPT0gaW5zdFsieHlfc2hhMjU2Il0gYW5kIHJlY29yZFsic2VlZHMiXSA9PSBTRUVEUwogICAgYXNzZXJ0IHJlY29yZFsiYnVkZ2V0X3MiXSA9PSBpbnN0WyJidWRnZXRfcyJdIGFuZCByZWNvcmRbImJ1aWxkIl0gPT0gQlVJTERfSU5GTwogICAgYXNzZXJ0IHJlY29yZFsiY29kZV9zaGEyNTYiXSA9PSBDT0RFX1NIQTI1NgogICAgYXNzZXJ0IHJlY29yZFsiZnJhY3Rpb24iXSA9PSBTV0FQX0ZSQUNUSU9OIGFuZCByZWNvcmRbImdyYXBoX2siXSA9PSBHUkFQSF9LCiAgICBhc3NlcnQgcmVjb3JkWyJoYXJkd2FyZV9wcm9maWxlIl0gPT0gSEFSRFdBUkVfUFJPRklMRQogICAgYXNzZXJ0IGxlbihyZWNvcmRbInNlZWRfY29zdHMiXSkgPT0gbGVuKFNFRURTKQogICAgYXNzZXJ0IG1pbihyZWNvcmRbInNlZWRfY29zdHMiXSkgPT0gcmVjb3JkWyJiZXN0X2Nvc3QiXQogICAgaWYgYXJtID09ICJuYXRpdmVfcG9wNiI6CiAgICAgICAgYXNzZXJ0IHNldChyZWNvcmRbImZpbmFsX3RvdXJfc291cmNlcyJdKSA9PSB7c3RyKHNlZWQpIGZvciBzZWVkIGluIFNFRURTfQogICAgZWxzZToKICAgICAgICBhc3NlcnQgbGVuKHJlY29yZFsiZmlyc3RfdG91cl9oYXNoZXMiXSkgPT0gbGVuKFNFRURTKQogICAgICAgIGFzc2VydCBsZW4ocmVjb3JkWyJsaXZlX2NhbmRpZGF0ZV9oYXNoZXMiXSkgPT0gbGVuKFNFRURTKQogICAgICAgIGFzc2VydCBsZW4ocmVjb3JkWyJuYXRpdmVfY2FuZGlkYXRlX2hhc2hlcyJdKSA9PSBsZW4oU0VFRFMpCiAgICAgICAgYXNzZXJ0IGxlbihyZWNvcmRbInByb3RlY3Rpb25fc2hhMjU2Il0pID09IDY0CiAgICAgICAgYXVkaXQgPSByZWNvcmRbInBvbGljeV9hdWRpdCJdCiAgICAgICAgYXNzZXJ0IGxlbihhdWRpdFsiZ2F0ZV9zaGEyNTZfcGVyX3NlZWQiXSkgPT0gbGVuKFNFRURTKQogICAgICAgIGFzc2VydCBhdWRpdFsic2VlZF9pZHMiXSA9PSBTRUVEUwogICAgICAgIGFzc2VydCBhdWRpdFsicG9saWN5X3ZlcnNpb24iXSA9PSBQT0xJQ1lfVkVSU0lPTgogICAgICAgIGFzc2VydCBhdWRpdFsiZGVjaXNpb25fYmFzaXMiXSBpbiAoImZyZXNoX2luZmVyZW5jZSIsICJjYWNoZWRfaW5mZXJlbmNlIikKICAgICAgICBpZiBhcm0gPT0gInNoYW0iOgogICAgICAgICAgICBhc3NlcnQgYXVkaXRbInNlbGVjdGVkX2FjdGlvbnNfcGVyX3NlZWQiXSA9PSBbMF0gKiBsZW4oU0VFRFMpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgYXNzZXJ0IGF1ZGl0WyJzZWxlY3RlZF9hY3Rpb25zX3Blcl9zZWVkIl0gPT0gYXVkaXRbImd1aWRlZF9hY3Rpb25zX3Blcl9zZWVkIl0KICAgICAgICBhc3NlcnQgYXVkaXRbInNlbGVjdGVkX3VuaXF1ZV91bmRpcmVjdGVkX2VkZ2VzX3Blcl9zZWVkIl0gPT0gYXVkaXRbInNlbGVjdGVkX2FjdGlvbnNfcGVyX3NlZWQiXQogICAgcmV0dXJuIHJlY29yZAoKCmRlZiBzYXZlX3Byb2dyZXNzKHN0YXRlPSJydW5uaW5nIiwgbGFzdD1Ob25lKToKICAgIGNvbXBsZXRlZCA9IHN1bShhcm1fcmVjb3JkX3BhdGgoaW5zdCwgYXJtKS5leGlzdHMoKSBmb3IgaW5zdCBpbiBJTlNUQU5DRVMgZm9yIGFybSBpbiBBUk1TKQogICAgd3JpdGVfanNvbl9hdG9taWMoT1VUUFVUX1JPT1QgLyAicHJvZ3Jlc3MuanNvbiIsCiAgICAgICAgICAgICAgICAgICAgICB7InN0YXRlIjogc3RhdGUsICJjb21wbGV0ZWRfYXJtcyI6IGNvbXBsZXRlZCwKICAgICAgICAgICAgICAgICAgICAgICAidG90YWxfYXJtcyI6IGxlbihJTlNUQU5DRVMpICogbGVuKEFSTVMpLCAibGFzdCI6IGxhc3QsCiAgICAgICAgICAgICAgICAgICAgICAgInVwZGF0ZWRfdXRjIjogdGltZS5zdHJmdGltZSgiJVktJW0tJWRUJUg6JU06JVNaIiwgdGltZS5nbXRpbWUoKSl9KQoKCm5ld19hcm1zID0gMApzYXZlX3Byb2dyZXNzKCkKZm9yIGluc3QgaW4gSU5TVEFOQ0VTOgogICAgaW5zdGFuY2VfZGlyID0gT1VUUFVUX1JPT1QgLyAiaW5zdGFuY2VzIiAvIGluc3RbIm5hbWUiXQogICAgaW5zdGFuY2VfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIGNvb3JkX3BhdGggPSBpbnN0YW5jZV9kaXIgLyAieHkubnB6IgogICAgaWYgbm90IGNvb3JkX3BhdGguZXhpc3RzKCk6CiAgICAgICAgdGVtcCA9IGluc3RhbmNlX2RpciAvICJ4eS50bXAubnB6IgogICAgICAgIG5wLnNhdmV6X2NvbXByZXNzZWQodGVtcCwgeHk9aW5zdFsieHkiXSkKICAgICAgICBvcy5yZXBsYWNlKHRlbXAsIGNvb3JkX3BhdGgpCiAgICB3aXRoIG5wLmxvYWQoY29vcmRfcGF0aCwgYWxsb3dfcGlja2xlPUZhbHNlKSBhcyBzYXZlZF94eToKICAgICAgICBhc3NlcnQgaGFzaGxpYi5zaGEyNTYobnAuYXNjb250aWd1b3VzYXJyYXkoc2F2ZWRfeHlbInh5Il0pLnRvYnl0ZXMoKSkuaGV4ZGlnZXN0KCkgPT0gaW5zdFsieHlfc2hhMjU2Il0KICAgIGxvY2FsX3Byb2JsZW0gPSBXT1JLX1JPT1QgLyAicHJvYmxlbXMiIC8gKGluc3RbIm5hbWUiXSArICIudHNwIikKICAgIGxvY2FsX3Byb2JsZW0ucGFyZW50Lm1rZGlyKGV4aXN0X29rPVRydWUpCiAgICBULndyaXRlX3RzcChsb2NhbF9wcm9ibGVtLCBpbnN0WyJ4eSJdKQogICAgZm9yIGFybSBpbiBhcm1fb3JkZXIoaW5zdFsibmFtZSJdKToKICAgICAgICByZWNvcmRfcGF0aCA9IGFybV9yZWNvcmRfcGF0aChpbnN0LCBhcm0pCiAgICAgICAgaWYgcmVjb3JkX3BhdGguZXhpc3RzKCk6CiAgICAgICAgICAgIHZhbGlkYXRlX3NhdmVkKGluc3QsIGFybSwganNvbi5sb2FkcyhyZWNvcmRfcGF0aC5yZWFkX3RleHQoKSkpCiAgICAgICAgICAgIHByaW50KCJBVExBOiIsIGluc3RbIm5hbWUiXSwgYXJtLCBmbHVzaD1UcnVlKQogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGlmIE1BWF9ORVdfQVJNUyBpcyBub3QgTm9uZSBhbmQgbmV3X2FybXMgPj0gTUFYX05FV19BUk1TOgogICAgICAgICAgICBicmVhawogICAgICAgIGF0dGVtcHRfaWQgPSB1dWlkLnV1aWQ0KCkuaGV4CiAgICAgICAgbG9jYWxfcnVuID0gV09SS19ST09UIC8gInJ1bnMiIC8gaW5zdFsibmFtZSJdIC8gYXJtIC8gYXR0ZW1wdF9pZAogICAgICAgIGxvY2FsX3J1bi5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgICAgIHRyeToKICAgICAgICAgICAgcHJpbnQoIkJBxZ5MQToiLCBpbnN0WyJuYW1lIl0sIGFybSwgImLDvHTDp2UiLCBpbnN0WyJidWRnZXRfcyJdLCAic24iLCBmbHVzaD1UcnVlKQogICAgICAgICAgICBpZiBhcm0uc3RhcnRzd2l0aCgibmF0aXZlXyIpOgogICAgICAgICAgICAgICAgcmF3ID0gcnVuX25hdGl2ZV9hcm0oYXJtLCBOQVRJVkVfTEtILCBsb2NhbF9wcm9ibGVtLCBpbnN0WyJ4eSJdLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgbG9jYWxfcnVuLCBTRUVEUywgaW5zdFsiYnVkZ2V0X3MiXSwgVCkKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgICMgQXJtcyB3aG9zZSBtb2RlbCBpbnB1dCBpcyBiaXR3aXNlIGlkZW50aWNhbCBkZWNpZGUgZnJvbSB0aGUKICAgICAgICAgICAgICAgICMgZmlyc3Qgc3RvcmVkIG1vZGVsIG91dHB1dDsgZXZlcnkgYXJtIHN0aWxsIHJ1bnMgaW5mZXJlbmNlLgogICAgICAgICAgICAgICAgcmF3ID0gcnVuX2d1aWRlZF9hcm0oYXJtLCBQQVRDSEVEX0xLSCwgbG9jYWxfcHJvYmxlbSwgaW5zdFsieHkiXSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGxvY2FsX3J1biwgU0VFRFMsIGluc3RbImJ1ZGdldF9zIl0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBNT0RFTCwgREVWSUNFLCBULCBNLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZnJhY3Rpb249U1dBUF9GUkFDVElPTiwgZ3JhcGhfaz1HUkFQSF9LLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZGVjaXNpb25fY2FjaGVfZGlyPWluc3RhbmNlX2RpciAvICJkZWNpc2lvbl9jYWNoZSIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBtb2RlbF9pZGVudGl0eT1DSEVDS1BPSU5UX1NIQTI1NikKICAgICAgICAgICAgYXVkaXRfcmVsID0gZiJpbnN0YW5jZXMve2luc3RbJ25hbWUnXX0ve2FybX0ve2F0dGVtcHRfaWR9IgogICAgICAgICAgICBhdWRpdF9kZXN0ID0gT1VUUFVUX1JPT1QgLyBhdWRpdF9yZWwKICAgICAgICAgICAgc2h1dGlsLmNvcHl0cmVlKGxvY2FsX3J1biwgYXVkaXRfZGVzdCkKICAgICAgICAgICAgcmVjb3JkID0geyJzY2hlbWEiOiAxLCAiY29tcGxldGUiOiBUcnVlLCAiaW5zdGFuY2UiOiBpbnN0WyJuYW1lIl0sCiAgICAgICAgICAgICAgICAgICAgICAiZ3JvdXAiOiBpbnN0WyJncm91cCJdLCAibiI6IGluc3RbIm4iXSwKICAgICAgICAgICAgICAgICAgICAgICJ4eV9zaGEyNTYiOiBpbnN0WyJ4eV9zaGEyNTYiXSwgImFybSI6IGFybSwKICAgICAgICAgICAgICAgICAgICAgICJzZWVkcyI6IFNFRURTLCAiYnVkZ2V0X3MiOiBpbnN0WyJidWRnZXRfcyJdLAogICAgICAgICAgICAgICAgICAgICAgInNlZWRfY29zdHMiOiByYXdbInNlZWRfY29zdHMiXSwgImJlc3RfY29zdCI6IHJhd1siYmVzdF9jb3N0Il0sCiAgICAgICAgICAgICAgICAgICAgICAid2FsbF9lbGFwc2VkX3MiOiByYXdbIndhbGxfZWxhcHNlZF9zIl0sCiAgICAgICAgICAgICAgICAgICAgICAicGhhc2VfdGltaW5nc19zIjogcmF3LmdldCgicGhhc2VfdGltaW5nc19zIiksCiAgICAgICAgICAgICAgICAgICAgICAicmVtYWluaW5nX2F0X3BvbGljeV9lbmRfcyI6IHJhdy5nZXQoInJlbWFpbmluZ19hdF9wb2xpY3lfZW5kX3MiKSwKICAgICAgICAgICAgICAgICAgICAgICJzdGFydF9lcG9jaF9zIjogcmF3WyJzdGFydF9lcG9jaF9zIl0sCiAgICAgICAgICAgICAgICAgICAgICAiZGVhZGxpbmVfZXBvY2hfcyI6IHJhd1siZGVhZGxpbmVfZXBvY2hfcyJdIGlmICJkZWFkbGluZV9lcG9jaF9zIiBpbiByYXcgZWxzZSByYXdbInN0YXJ0X2Vwb2NoX3MiXSArIGluc3RbImJ1ZGdldF9zIl0sCiAgICAgICAgICAgICAgICAgICAgICAiZmlyc3RfY29zdHMiOiByYXcuZ2V0KCJmaXJzdF9jb3N0cyIpLAogICAgICAgICAgICAgICAgICAgICAgImZpcnN0X3RvdXJfaGFzaGVzIjogcmF3LmdldCgiZmlyc3RfdG91cl9oYXNoZXMiKSwKICAgICAgICAgICAgICAgICAgICAgICJsaXZlX2NhbmRpZGF0ZV9oYXNoZXMiOiByYXcuZ2V0KCJsaXZlX2NhbmRpZGF0ZV9oYXNoZXMiKSwKICAgICAgICAgICAgICAgICAgICAgICJuYXRpdmVfY2FuZGlkYXRlX2hhc2hlcyI6IHJhdy5nZXQoIm5hdGl2ZV9jYW5kaWRhdGVfaGFzaGVzIiksCiAgICAgICAgICAgICAgICAgICAgICAicHJvdGVjdGlvbl9zaGEyNTYiOiByYXcuZ2V0KCJwcm90ZWN0aW9uX3NoYTI1NiIpLAogICAgICAgICAgICAgICAgICAgICAgInBvbGljeV9hdWRpdCI6IHJhdy5nZXQoInBvbGljeV9hdWRpdCIpLAogICAgICAgICAgICAgICAgICAgICAgImd1aWRlZF9pbXByb3ZlbWVudF9ldmVudHMiOiByYXcuZ2V0KCJndWlkZWRfaW1wcm92ZW1lbnRfZXZlbnRzIiksCiAgICAgICAgICAgICAgICAgICAgICAibmF0aXZlX2ltcHJvdmVtZW50X2V2ZW50cyI6IHJhdy5nZXQoIm5hdGl2ZV9pbXByb3ZlbWVudF9ldmVudHMiKSwKICAgICAgICAgICAgICAgICAgICAgICJmaW5hbF90b3VyX3NvdXJjZXMiOiByYXcuZ2V0KCJmaW5hbF90b3VyX3NvdXJjZXMiKSwKICAgICAgICAgICAgICAgICAgICAgICJldmVudF9mbGFncyI6IHJhdy5nZXQoImV2ZW50X2ZsYWdzIiksCiAgICAgICAgICAgICAgICAgICAgICAiZGVjaXNpb25fY2FjaGVfZmlsZXNfd3JpdHRlbiI6IHJhdy5nZXQoImRlY2lzaW9uX2NhY2hlX2ZpbGVzX3dyaXR0ZW4iKSwKICAgICAgICAgICAgICAgICAgICAgICJmcmFjdGlvbiI6IFNXQVBfRlJBQ1RJT04sICJncmFwaF9rIjogR1JBUEhfSywKICAgICAgICAgICAgICAgICAgICAgICJidWlsZCI6IEJVSUxEX0lORk8sICJjb2RlX3NoYTI1NiI6IENPREVfU0hBMjU2LAogICAgICAgICAgICAgICAgICAgICAgImhhcmR3YXJlIjogSEFSRFdBUkUsICJoYXJkd2FyZV9wcm9maWxlIjogSEFSRFdBUkVfUFJPRklMRSwKICAgICAgICAgICAgICAgICAgICAgICJhdWRpdF9yZWxwYXRoIjogYXVkaXRfcmVsLAogICAgICAgICAgICAgICAgICAgICAgImF0dGVtcHRfaWQiOiBhdHRlbXB0X2lkLAogICAgICAgICAgICAgICAgICAgICAgImNvbXBsZXRlZF91dGMiOiB0aW1lLnN0cmZ0aW1lKCIlWS0lbS0lZFQlSDolTTolU1oiLCB0aW1lLmdtdGltZSgpKX0KICAgICAgICAgICAgdmFsaWRhdGVfc2F2ZWQoaW5zdCwgYXJtLCByZWNvcmQpCiAgICAgICAgICAgIHdyaXRlX2pzb25fYXRvbWljKHJlY29yZF9wYXRoLCByZWNvcmQpCiAgICAgICAgICAgIG5ld19hcm1zICs9IDEKICAgICAgICAgICAgc2F2ZV9wcm9ncmVzcyhsYXN0PWluc3RbIm5hbWUiXSArICIvIiArIGFybSkKICAgICAgICAgICAgcHJpbnQoIkLEsFRUxLA6IiwgaW5zdFsibmFtZSJdLCBhcm0sICJlbiBpeWkiLCByYXdbImJlc3RfY29zdCJdLCBmbHVzaD1UcnVlKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOgogICAgICAgICAgICBmYWlsdXJlX2RpciA9IE9VVFBVVF9ST09UIC8gImVycm9ycyIgLyAoaW5zdFsibmFtZSJdICsgIl8iICsgYXJtICsgIl8iICsgYXR0ZW1wdF9pZCkKICAgICAgICAgICAgZmFpbHVyZV9kaXIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgICAgICAgICBhdWRpdF9jb3B5X2Vycm9yID0gTm9uZQogICAgICAgICAgICBpZiBsb2NhbF9ydW4uZXhpc3RzKCk6CiAgICAgICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICAgICAgc2h1dGlsLmNvcHl0cmVlKGxvY2FsX3J1biwgZmFpbHVyZV9kaXIgLyAiYXVkaXQiKQogICAgICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBjb3B5X2V4YzoKICAgICAgICAgICAgICAgICAgICBhdWRpdF9jb3B5X2Vycm9yID0gcmVwcihjb3B5X2V4YykKICAgICAgICAgICAgd3JpdGVfanNvbl9hdG9taWMoZmFpbHVyZV9kaXIgLyAiZXJyb3IuanNvbiIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHsiaW5zdGFuY2UiOiBpbnN0WyJuYW1lIl0sICJhcm0iOiBhcm0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAiYXR0ZW1wdF9pZCI6IGF0dGVtcHRfaWQsICJidWRnZXRfcyI6IGluc3RbImJ1ZGdldF9zIl0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAiaGFyZHdhcmVfcHJvZmlsZSI6IEhBUkRXQVJFX1BST0ZJTEUsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAiZXJyb3IiOiByZXByKGV4YyksICJhdWRpdF9jb3B5X2Vycm9yIjogYXVkaXRfY29weV9lcnJvciwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJ1dGMiOiB0aW1lLnN0cmZ0aW1lKCIlWS0lbS0lZFQlSDolTTolU1oiLCB0aW1lLmdtdGltZSgpKX0pCiAgICAgICAgICAgIHByaW50KCJCYcWfYXLEsXPEsXogZGVuZW1lIGthecSxdGxhcsSxOiIsIGZhaWx1cmVfZGlyLCBmbHVzaD1UcnVlKQogICAgICAgICAgICBzYXZlX3Byb2dyZXNzKHN0YXRlPSJlcnJvciIsIGxhc3Q9aW5zdFsibmFtZSJdICsgIi8iICsgYXJtKQogICAgICAgICAgICByYWlzZQogICAgaWYgTUFYX05FV19BUk1TIGlzIG5vdCBOb25lIGFuZCBuZXdfYXJtcyA+PSBNQVhfTkVXX0FSTVM6CiAgICAgICAgYnJlYWsKCmNvbXBsZXRlZF9jb3VudCA9IHN1bShhcm1fcmVjb3JkX3BhdGgoaW5zdCwgYXJtKS5leGlzdHMoKSBmb3IgaW5zdCBpbiBJTlNUQU5DRVMgZm9yIGFybSBpbiBBUk1TKQpzYXZlX3Byb2dyZXNzKHN0YXRlPSJjb21wbGV0ZSIgaWYgY29tcGxldGVkX2NvdW50ID09IGxlbihJTlNUQU5DRVMpICogbGVuKEFSTVMpIGVsc2UgInBhcnRpYWwiKQpwcmludChmIlRhbWFtbGFuYW4ga29sOiB7Y29tcGxldGVkX2NvdW50fS97bGVuKElOU1RBTkNFUykgKiBsZW4oQVJNUyl9LiAiCiAgICAgIGYiQnUgb3R1cnVtZGFraSB5ZW5pIGtvbDoge25ld19hcm1zfSIpCg==', 'v15_live_swap/analysis_cell.py': 'IiIidjE1IGxpdmUtc3dhcCBleHBsb3JhdG9yeSBhbmFseXNpcyBvbiBwYWlyZWQgZGV2ZWxvcG1lbnQgaW5zdGFuY2VzLgoKT25seSBjb21wbGV0ZSBzYW1lLWhhcmR3YXJlIGluc3RhbmNlcyBjb250cmlidXRlIHRvIGVuZHBvaW50IGNvbXBhcmlzb25zLgpQb2xpY3kgY29tcGFyaXNvbnMgYWRkaXRpb25hbGx5IHJlcXVpcmUgZXhhY3QgZmlyc3QgdG91cnMsIGNhbmRpZGF0ZSBzdGF0ZXMsCm1vZGVsIGRlY2lzaW9uIGFycmF5cywgc291cmNlIGdhdGVzLCB2aWN0aW1zIGFuZCBhY3Rpb24gY2FwYWNpdGllcy4gIFRoZSBmb3VyCmluc3RhbmNlcyB3ZXJlIGV4YW1pbmVkIGluIHByZXZpb3VzIHJ1bnMsIHNvIHAtdmFsdWVzIGFyZSBkZXNjcmlwdGl2ZSBvbmx5LgoiIiIKCmZyb20gbWF0aCBpbXBvcnQgY29tYgoKCmRlZiByZWFkX2tleV92YWx1ZV9tZXRyaWNzKHBhdGgpOgogICAgdmFsdWVzID0ge30KICAgIGZvciBsaW5lIGluIFBhdGgocGF0aCkucmVhZF90ZXh0KCkuc3BsaXRsaW5lcygpOgogICAgICAgIHdvcmRzID0gbGluZS5zcGxpdCgpCiAgICAgICAgaWYgbGVuKHdvcmRzKSAhPSAyIG9yIHdvcmRzWzBdIGluIHZhbHVlczoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkJvenVrIHRlbGVtZXRyaSBzYXTEsXLEsToge3BhdGh9OiB7bGluZSFyfSIpCiAgICAgICAgdmFsdWVzW3dvcmRzWzBdXSA9IGZsb2F0KHdvcmRzWzFdKSBpZiAiLiIgaW4gd29yZHNbMV0gZWxzZSBpbnQod29yZHNbMV0pCiAgICByZXR1cm4gdmFsdWVzCgoKZGVmIGV4YWN0X3NpZ25fcCh3aW5zLCBsb3NzZXMpOgogICAgbiwgayA9IHdpbnMgKyBsb3NzZXMsIG1pbih3aW5zLCBsb3NzZXMpCiAgICByZXR1cm4gMS4wIGlmIG4gPT0gMCBlbHNlIG1pbigxLjAsIDIuMCAqIHN1bShjb21iKG4sIGopIGZvciBqIGluIHJhbmdlKGsgKyAxKSkgLyAyICoqIG4pCgoKZGVmIGNvbXBhcmUoZnJhbWUsIHRyZWF0bWVudCwgY29udHJvbCwgYmFzaXMsIGNvbHVtbj0ie30iKToKICAgIGlmIGZyYW1lLmVtcHR5OgogICAgICAgIHJldHVybiBOb25lCiAgICB0ID0gZnJhbWVbY29sdW1uLmZvcm1hdCh0cmVhdG1lbnQpXS50b19udW1weShkdHlwZT1ucC5mbG9hdDY0KQogICAgYyA9IGZyYW1lW2NvbHVtbi5mb3JtYXQoY29udHJvbCldLnRvX251bXB5KGR0eXBlPW5wLmZsb2F0NjQpCiAgICByZWwgPSAxMDAuMCAqICh0IC0gYykgLyBjCiAgICB3aW5zLCBsb3NzZXMgPSBpbnQoKHJlbCA8IDApLnN1bSgpKSwgaW50KChyZWwgPiAwKS5zdW0oKSkKICAgIHJldHVybiB7InRyZWF0bWVudCI6IHRyZWF0bWVudCwgImNvbnRyb2wiOiBjb250cm9sLCAiYmFzaXMiOiBiYXNpcywKICAgICAgICAgICAgIm5faW5zdGFuY2VzIjogbGVuKHJlbCksICJ3aW5zIjogd2lucywgImxvc3NlcyI6IGxvc3NlcywKICAgICAgICAgICAgInRpZXMiOiBpbnQoKHJlbCA9PSAwKS5zdW0oKSksCiAgICAgICAgICAgICJtZWFuX3JlbGF0aXZlX3BjdCI6IGZsb2F0KHJlbC5tZWFuKCkpLAogICAgICAgICAgICAibWVkaWFuX3JlbGF0aXZlX3BjdCI6IGZsb2F0KG5wLm1lZGlhbihyZWwpKSwKICAgICAgICAgICAgImV4YWN0X3NpZ25fcF9kZXNjcmlwdGl2ZSI6IGV4YWN0X3NpZ25fcCh3aW5zLCBsb3NzZXMpfQoKCnBhdGNoZWRfYXJtcyA9IFsic2hhbSIsICJyYW5kb21fc3dhcCIsICJuZWFyZXN0X3N3YXAiLCAiYmV0YV9zd2FwIl0KZ3VpZGVkX2FybXMgPSBwYXRjaGVkX2FybXNbMTpdCnJvd3MsIHNlZWRfcm93cywgbWVjaGFuaXNtX3Jvd3MgPSBbXSwgW10sIFtdCnN1bW1hcnlfZGlyID0gT1VUUFVUX1JPT1QgLyAic3VtbWFyeSIKc3VtbWFyeV9kaXIubWtkaXIoZXhpc3Rfb2s9VHJ1ZSkKCmZvciBpbnN0IGluIElOU1RBTkNFUzoKICAgIHJlY29yZHMgPSB7fQogICAgZm9yIGFybSBpbiBBUk1TOgogICAgICAgIHBhdGggPSBhcm1fcmVjb3JkX3BhdGgoaW5zdCwgYXJtKQogICAgICAgIGlmIHBhdGguZXhpc3RzKCk6CiAgICAgICAgICAgIHJlY29yZHNbYXJtXSA9IHZhbGlkYXRlX3NhdmVkKGluc3QsIGFybSwganNvbi5sb2FkcyhwYXRoLnJlYWRfdGV4dCgpKSkKICAgIGlmIGxlbihyZWNvcmRzKSAhPSBsZW4oQVJNUyk6CiAgICAgICAgY29udGludWUKICAgIGF1ZGl0cyA9IHthcm06IHJlY29yZHNbYXJtXVsicG9saWN5X2F1ZGl0Il0gZm9yIGFybSBpbiBwYXRjaGVkX2FybXN9CiAgICByb3cgPSB7Imluc3RhbmNlIjogaW5zdFsibmFtZSJdLCAiZ3JvdXAiOiBpbnN0WyJncm91cCJdLAogICAgICAgICAgICJuIjogaW5zdFsibiJdLCAiYnVkZ2V0X3MiOiBpbnN0WyJidWRnZXRfcyJdLAogICAgICAgICAgICJtaXhlZF9oYXJkd2FyZSI6IGxlbih7anNvbi5kdW1wcyhyZWNvcmRzW2FybV1bImhhcmR3YXJlX3Byb2ZpbGUiXSwgc29ydF9rZXlzPVRydWUpCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBmb3IgYXJtIGluIEFSTVN9KSAhPSAxfQogICAgZm9yIGFybSBpbiBBUk1TOgogICAgICAgIHJvd1thcm1dID0gcmVjb3Jkc1thcm1dWyJiZXN0X2Nvc3QiXQogICAgICAgIHJvd1sibWVhbjZfIiArIGFybV0gPSBmbG9hdChucC5tZWFuKHJlY29yZHNbYXJtXVsic2VlZF9jb3N0cyJdKSkKICAgIHJvd1siZmlyc3RfdG91cnNfbWF0Y2giXSA9IGFsbCgKICAgICAgICBsZW4oe3JlY29yZHNbYXJtXVsiZmlyc3RfdG91cl9oYXNoZXMiXVtpXSBmb3IgYXJtIGluIHBhdGNoZWRfYXJtc30pID09IDEKICAgICAgICBmb3IgaSBpbiByYW5nZShsZW4oU0VFRFMpKSkKICAgIHJvd1sicmF3X2ZpcnN0X3RvdXJzX21hdGNoIl0gPSBhbGwoCiAgICAgICAgbGVuKHthdWRpdHNbYXJtXVsiZmlyc3RfdG91cl9yYXdfc2hhMjU2Il1baV0gZm9yIGFybSBpbiBwYXRjaGVkX2FybXN9KSA9PSAxCiAgICAgICAgZm9yIGkgaW4gcmFuZ2UobGVuKFNFRURTKSkpCiAgICByb3dbIm5hdGl2ZV9jYW5kaWRhdGVzX21hdGNoIl0gPSBhbGwoCiAgICAgICAgbGVuKHtyZWNvcmRzW2FybV1bIm5hdGl2ZV9jYW5kaWRhdGVfaGFzaGVzIl1baV0gZm9yIGFybSBpbiBwYXRjaGVkX2FybXN9KSA9PSAxCiAgICAgICAgZm9yIGkgaW4gcmFuZ2UobGVuKFNFRURTKSkpCiAgICByb3dbImxpdmVfY2FuZGlkYXRlc19tYXRjaCJdID0gYWxsKAogICAgICAgIGxlbih7cmVjb3Jkc1thcm1dWyJsaXZlX2NhbmRpZGF0ZV9oYXNoZXMiXVtpXSBmb3IgYXJtIGluIHBhdGNoZWRfYXJtc30pID09IDEKICAgICAgICBmb3IgaSBpbiByYW5nZShsZW4oU0VFRFMpKSkKICAgIHJvd1sicHJvdGVjdGVkX2VkZ2VzX21hdGNoIl0gPSBsZW4oe3JlY29yZHNbYXJtXVsicHJvdGVjdGlvbl9zaGEyNTYiXQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGZvciBhcm0gaW4gcGF0Y2hlZF9hcm1zfSkgPT0gMQogICAgcm93WyJnYXRlc19tYXRjaCJdID0gYWxsKAogICAgICAgIGxlbih7YXVkaXRzW2FybV1bImdhdGVfc2hhMjU2X3Blcl9zZWVkIl1baV0gZm9yIGFybSBpbiBwYXRjaGVkX2FybXN9KSA9PSAxCiAgICAgICAgZm9yIGkgaW4gcmFuZ2UobGVuKFNFRURTKSkpCiAgICByb3dbInZpY3RpbXNfbWF0Y2giXSA9IGFsbCgKICAgICAgICBsZW4oe2F1ZGl0c1thcm1dWyJ2aWN0aW1fc2hhMjU2X3Blcl9zZWVkIl1baV0gZm9yIGFybSBpbiBwYXRjaGVkX2FybXN9KSA9PSAxCiAgICAgICAgZm9yIGkgaW4gcmFuZ2UobGVuKFNFRURTKSkpCiAgICByb3dbIm1vZGVsX3ByZWRpY3Rpb25zX21hdGNoIl0gPSBsZW4oe2F1ZGl0c1thcm1dWyJtb2RlbF9wcm9iYWJpbGl0eV9zaGEyNTYiXQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZm9yIGFybSBpbiBwYXRjaGVkX2FybXN9KSA9PSAxCiAgICByb3dbInBsYW5uZWRfYWN0aW9uc19tYXRjaCJdID0gbGVuKHsKICAgICAgICBqc29uLmR1bXBzKGF1ZGl0c1thcm1dWyJhbGxfZ3VpZGVkX2FjdGlvbl9zaGEyNTZfcGVyX3NlZWQiXSwgc29ydF9rZXlzPVRydWUpCiAgICAgICAgZm9yIGFybSBpbiBwYXRjaGVkX2FybXN9KSA9PSAxCiAgICByb3dbImNhcGFjaXRpZXNfbWF0Y2giXSA9IGFsbCgKICAgICAgICBsZW4oe2F1ZGl0c1thcm1dWyJzZWxlY3RlZF9hY3Rpb25zX3Blcl9zZWVkIl1baV0gZm9yIGFybSBpbiBndWlkZWRfYXJtc30pID09IDEKICAgICAgICBmb3IgaSBpbiByYW5nZShsZW4oU0VFRFMpKSkKICAgIHJvd1sibWF0Y2hlZF9wb2xpY3lfc3RhdGUiXSA9IGFsbChyb3dba2V5XSBmb3Iga2V5IGluICgKICAgICAgICAiZmlyc3RfdG91cnNfbWF0Y2giLCAicmF3X2ZpcnN0X3RvdXJzX21hdGNoIiwgIm5hdGl2ZV9jYW5kaWRhdGVzX21hdGNoIiwKICAgICAgICAibGl2ZV9jYW5kaWRhdGVzX21hdGNoIiwgInByb3RlY3RlZF9lZGdlc19tYXRjaCIsICJnYXRlc19tYXRjaCIsCiAgICAgICAgInZpY3RpbXNfbWF0Y2giLCAibW9kZWxfcHJlZGljdGlvbnNfbWF0Y2giLCAicGxhbm5lZF9hY3Rpb25zX21hdGNoIiwKICAgICAgICAiY2FwYWNpdGllc19tYXRjaCIpKQogICAgcm93cy5hcHBlbmQocm93KQoKICAgIGZvciBpLCBzZWVkIGluIGVudW1lcmF0ZShTRUVEUyk6CiAgICAgICAgc2VlZF9yb3dzLmFwcGVuZCh7Imluc3RhbmNlIjogaW5zdFsibmFtZSJdLCAic2VlZCI6IHNlZWQsCiAgICAgICAgICAgICAgICAgICAgICAgICAgKip7YXJtOiByZWNvcmRzW2FybV1bInNlZWRfY29zdHMiXVtpXSBmb3IgYXJtIGluIEFSTVN9fSkKICAgIGZvciBhcm0gaW4gcGF0Y2hlZF9hcm1zOgogICAgICAgIGZvciBpLCBzZWVkIGluIGVudW1lcmF0ZShTRUVEUyk6CiAgICAgICAgICAgIGJhc2UgPSBPVVRQVVRfUk9PVCAvIHJlY29yZHNbYXJtXVsiYXVkaXRfcmVscGF0aCJdIC8gZiJzZWVkX3tzZWVkfSIKICAgICAgICAgICAgbWV0cmljcyA9IHJlYWRfa2V5X3ZhbHVlX21ldHJpY3MoYmFzZSAvICJ0ZWxlbWV0cnkudHh0IikKICAgICAgICAgICAgYXBwbGllZCA9IHJlYWRfa2V5X3ZhbHVlX21ldHJpY3MoYmFzZSAvICJhcHBsaWVkLnN0YXRzIikKICAgICAgICAgICAgc2VsZWN0ZWQgPSBpbnQoYXVkaXRzW2FybV1bInNlbGVjdGVkX2FjdGlvbnNfcGVyX3NlZWQiXVtpXSkKICAgICAgICAgICAgYXNzZXJ0IGludChtZXRyaWNzWyJhY3Rpb25fY291bnQiXSkgPT0gc2VsZWN0ZWQKICAgICAgICAgICAgYXNzZXJ0IGludChhcHBsaWVkWyJzd2FwcGVkIl0pID09IHNlbGVjdGVkCiAgICAgICAgICAgIGFzc2VydCBpbnQoYXBwbGllZFsicmVqZWN0ZWQiXSkgPT0gMAogICAgICAgICAgICBhc3NlcnQgaW50KGFwcGxpZWRbInJvd19sZW5ndGhfcHJlc2VydmVkIl0pID09IDEKICAgICAgICAgICAgbWVjaGFuaXNtX3Jvd3MuYXBwZW5kKHsiaW5zdGFuY2UiOiBpbnN0WyJuYW1lIl0sICJhcm0iOiBhcm0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgInNlZWQiOiBzZWVkLCAic2VsZWN0ZWRfYWN0aW9ucyI6IHNlbGVjdGVkLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJiYXJyaWVyX3dhaXRfcyI6IGZsb2F0KGFwcGxpZWRbIndhaXRfd2FsbF9zIl0pLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJ0ZWxlbWV0cnlfYWdlX2F0X2VuZF9zIjogbWF4KDAuMCwgbWluKAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBmbG9hdChyZWNvcmRzW2FybV1bImRlYWRsaW5lX2Vwb2NoX3MiXSksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGZsb2F0KHJlY29yZHNbYXJtXVsic3RhcnRfZXBvY2hfcyJdKSArCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGZsb2F0KHJlY29yZHNbYXJtXVsid2FsbF9lbGFwc2VkX3MiXSkpIC0KICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZmxvYXQobWV0cmljc1sidGVsZW1ldHJ5X3NuYXBzaG90X2Vwb2NoX3MiXSkpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICoqbWV0cmljc30pCgp0YWJsZSA9IHBkLkRhdGFGcmFtZShyb3dzKQpzZWVkX3RhYmxlID0gcGQuRGF0YUZyYW1lKHNlZWRfcm93cykKbWVjaGFuaXNtX3RhYmxlID0gcGQuRGF0YUZyYW1lKG1lY2hhbmlzbV9yb3dzKQp0YWJsZS50b19jc3Yoc3VtbWFyeV9kaXIgLyAiY29tcGxldGVfaW5zdGFuY2VzLmNzdiIsIGluZGV4PUZhbHNlKQpzZWVkX3RhYmxlLnRvX2NzdihzdW1tYXJ5X2RpciAvICJzZWVkX2Nvc3RzLmNzdiIsIGluZGV4PUZhbHNlKQptZWNoYW5pc21fdGFibGUudG9fY3N2KHN1bW1hcnlfZGlyIC8gInN3YXBfbWVjaGFuaXNtLmNzdiIsIGluZGV4PUZhbHNlKQoKc2FtZV9oYXJkd2FyZSA9IHRhYmxlLmxvY1t+dGFibGVbIm1peGVkX2hhcmR3YXJlIl1dLmNvcHkoKSBpZiBub3QgdGFibGUuZW1wdHkgZWxzZSB0YWJsZQptYXRjaGVkID0gc2FtZV9oYXJkd2FyZS5sb2Nbc2FtZV9oYXJkd2FyZVsibWF0Y2hlZF9wb2xpY3lfc3RhdGUiXV0uY29weSgpIGlmIG5vdCBzYW1lX2hhcmR3YXJlLmVtcHR5IGVsc2Ugc2FtZV9oYXJkd2FyZQpjb21wYXJpc29ucyA9IFtdCmZvciBjb250cm9sIGluICgibmF0aXZlX3BvcDYiLCAic2hhbSIsICJuZWFyZXN0X3N3YXAiLCAicmFuZG9tX3N3YXAiKToKICAgIGJhc2lzID0gInNhbWUgaGFyZHdhcmUsIGJlc3Qtb2Ytc2l4IGVuZHBvaW50IgogICAgZnJhbWUgPSBzYW1lX2hhcmR3YXJlIGlmIGNvbnRyb2wgPT0gIm5hdGl2ZV9wb3A2IiBlbHNlIG1hdGNoZWQKICAgIHJlc3VsdCA9IGNvbXBhcmUoZnJhbWUsICJiZXRhX3N3YXAiLCBjb250cm9sLCBiYXNpcykKICAgIGlmIHJlc3VsdDoKICAgICAgICBjb21wYXJpc29ucy5hcHBlbmQocmVzdWx0KQogICAgcmVzdWx0ID0gY29tcGFyZShmcmFtZSwgImJldGFfc3dhcCIsIGNvbnRyb2wsICJkZXNjcmlwdGl2ZSBtZWFuLW9mLXNpeCIsICJtZWFuNl97fSIpCiAgICBpZiByZXN1bHQ6CiAgICAgICAgY29tcGFyaXNvbnMuYXBwZW5kKHJlc3VsdCkKY29tcGFyaXNvbl90YWJsZSA9IHBkLkRhdGFGcmFtZShjb21wYXJpc29ucykKY29tcGFyaXNvbl90YWJsZS50b19jc3Yoc3VtbWFyeV9kaXIgLyAiZXhwbG9yYXRvcnlfY29tcGFyaXNvbnMuY3N2IiwgaW5kZXg9RmFsc2UpCgptZWNoYW5pc21fc3VtbWFyeSA9IHBkLkRhdGFGcmFtZSgpCmlmIG5vdCBtZWNoYW5pc21fdGFibGUuZW1wdHk6CiAgICBtZWFzdXJlcyA9IFtuYW1lIGZvciBuYW1lIGluICgKICAgICAgICAic2VsZWN0ZWRfYWN0aW9ucyIsICJhY3Rpb25fY2FuZGlkYXRlX2VudW1lcmF0aW9ucyIsCiAgICAgICAgImFjdGlvbl9hY2NlcHRlZF9lZGdlX2VudHJpZXMiLCAiYWN0aW9uX2luY3VtYmVudF9lbnRyaWVzIiwKICAgICAgICAidHJpYWxzX2NvbXBsZXRlZF9hZnRlcl9hY3Rpb24iLCAidG90YWxfY2FuZGlkYXRlX2VudW1lcmF0aW9ucyIpCiAgICAgICAgaWYgbmFtZSBpbiBtZWNoYW5pc21fdGFibGUuY29sdW1uc10KICAgIG1lY2hhbmlzbV9zdW1tYXJ5ID0gbWVjaGFuaXNtX3RhYmxlLmdyb3VwYnkoImFybSIsIGFzX2luZGV4PUZhbHNlKVttZWFzdXJlc10uc3VtKCkKbWVjaGFuaXNtX3N1bW1hcnkudG9fY3N2KHN1bW1hcnlfZGlyIC8gInN3YXBfbWVjaGFuaXNtX3RvdGFscy5jc3YiLCBpbmRleD1GYWxzZSkKCndyaXRlX2pzb25fYXRvbWljKHN1bW1hcnlfZGlyIC8gInBpbG90X25vdGUuanNvbiIsIHsKICAgICJjb21wbGV0ZWRfaW5zdGFuY2VzIjogbGVuKHRhYmxlKSwgInBsYW5uZWRfaW5zdGFuY2VzIjogbGVuKElOU1RBTkNFUyksCiAgICAic2FtZV9oYXJkd2FyZV9pbnN0YW5jZXMiOiBsZW4oc2FtZV9oYXJkd2FyZSksCiAgICAibWF0Y2hlZF9wb2xpY3lfaW5zdGFuY2VzIjogbGVuKG1hdGNoZWQpLAogICAgImludGVycHJldGF0aW9uIjogIlJldXNlZC1jYXNlIGV4cGxvcmF0b3J5IHBpbG90LiBJbnN0YW5jZSBpcyB0aGUgYW5hbHlzaXMgdW5pdDsgc2VlZHMgYXJlIG5vdCBpbmRlcGVuZGVudC4gU2lnbiBwLXZhbHVlcyBhcmUgZGVzY3JpcHRpdmUuIE5hdGl2ZSBQT1A2IGlzIHRoZSB1bmludGVycnVwdGVkIHByYWN0aWNhbCBiZW5jaG1hcmsuIERpcmVjdGVkIHJvdyBsZW5ndGggaXMgZml4ZWQgb25seSBhdCB0aGUgc3dhcCBiYXJyaWVyOyBMS0ggbWF5IGFwcGVuZCBvciByZW9yZGVyIGNhbmRpZGF0ZXMgbGF0ZXIuIFJldmVyc2Utcm93IHByZXNlbmNlIG9mIGV2aWN0ZWQgZWRnZXMgY2FuIGxlYXZlIHRoZSB1bmRpcmVjdGVkIGdyYXBoIHNpemUgdW5jaGFuZ2VkIG9yIGluY3JlYXNlIGl0LiBObyBzdXBlcmlvcml0eSBjbGFpbSBiZWZvcmUgZGlzam9pbnQgaG9sZG91dC4ifSkKCnByaW50KCJUYW1hbWxhbmFuIMO2cm5lazoiLCBsZW4odGFibGUpLCAiLyIsIGxlbihJTlNUQU5DRVMpLAogICAgICAiOyBlxZ9sZW5tacWfIHBvbGl0aWthIMO2cm5lxJ9pOiIsIGxlbihtYXRjaGVkKSkKZGlzcGxheSh0YWJsZSkKZGlzcGxheShjb21wYXJpc29uX3RhYmxlKQpkaXNwbGF5KG1lY2hhbmlzbV9zdW1tYXJ5KQo=', 'v15_live_swap/README.md': 'IyBCZXRhVG91ciB2MTVyMjogbGl2ZSBjYW5kaWRhdGUgcmVwbGFjZW1lbnQgcGlsb3QKClRoaXMgZXhwbG9yYXRvcnkgZXhwZXJpbWVudCB0ZXN0cyB3aGV0aGVyIEJldGFUb3VyIGNhbiBpbXByb3ZlIExLSC0zLjAuMTQgYnkgcmVwbGFjaW5nIGxvdy1wcmlvcml0eSBQT1BNVVNJQy02IGNhbmRpZGF0ZXMgKippbiBwbGFjZSoqIGFmdGVyIHRoZSBmaXJzdCBjb21wbGV0ZWQgdHJpYWwuIFVubGlrZSB0aGUgdjExcjIgY2FuZGlkYXRlLWFkZGl0aW9uIHBpbG90LCB0aGUgaW50ZXJ2ZW50aW9uIGRvZXMgbm90IGxlbmd0aGVuIGRpcmVjdGVkIGNhbmRpZGF0ZSByb3dzIGF0IHRoZSBiYXJyaWVyLiBMS0ggbWF5IGxhdGVyIGFwcGVuZCBvciByZW9yZGVyIHJvd3MgdGhyb3VnaCBpdHMgb3JkaW5hcnkgc2VhcmNoIGxvZ2ljLiBBIHZpY3RpbSBjYW4gcmVtYWluIGluIHRoZSByZXZlcnNlIHJvdywgc28gdGhlIGludGVydmVudGlvbiBpcyBub3QgZ3VhcmFudGVlZCB0byBrZWVwIHRoZSBudW1iZXIgb2YgdW5pcXVlIHVuZGlyZWN0ZWQgZWRnZXMgZml4ZWQuIFRoZSBjb2RlIHJlY29yZHMgdGhpcyBjb25kaXRpb24uCgpUaGUgZm91ciAxMCwwMDAtbm9kZSBpbnN0YW5jZXMgYXJlIHJldXNlZCBmcm9tIGVhcmxpZXIgcGlsb3RzLiBSZXN1bHRzIGFyZSBmb3IgcG9saWN5IGRldmVsb3BtZW50LCBub3QgYW4gaW5kZXBlbmRlbnQgcGVyZm9ybWFuY2UgY2xhaW0uIFRoZSBtYW51c2NyaXB0IGlzIHVuY2hhbmdlZCBwZW5kaW5nIGEgcG9zaXRpdmUgZGlzam9pbnQgaG9sZG91dC4KCiMjIEFybXMgYW5kIGZhaXJuZXNzCgp8IEFybSB8IEludGVydmVudGlvbiBhZnRlciBmaXJzdCB0cmlhbCB8CnwgLS0tIHwgLS0tIHwKfCBgbmF0aXZlX3BvcDZgIHwgVW5tb2RpZmllZCwgdW5pbnRlcnJ1cHRlZCBQT1BNVVNJQy02IExLSDsgcHJhY3RpY2FsIGJhc2VsaW5lLiB8CnwgYHNoYW1gIHwgUGF0Y2hlZCBMS0ggcGF1c2VzIGZvciB0aGUgYmFycmllciBhbmQgcGF5cyBmb3IgbW9kZWwgaW5mZXJlbmNlOyBubyBzd2FwLiB8CnwgYHJhbmRvbV9zd2FwYCB8IERldGVybWluaXN0aWNhbGx5IHJhbmRvbSBlbGlnaWJsZSBuZXcgbmVpZ2hib3IuIHwKfCBgbmVhcmVzdF9zd2FwYCB8IE5lYXJlc3QgZWxpZ2libGUgbmV3IG5laWdoYm9yLiB8CnwgYGJldGFfc3dhcGAgfCBIaWdoZXN0IG1vZGVsIHByb2JhYmlsaXR5IGFtb25nIGVsaWdpYmxlIG5ldyBuZWlnaGJvcnMuIHwKClRoZSB0aHJlZSBzd2FwIGFybXMgdXNlIGlkZW50aWNhbCBzb3VyY2Ugbm9kZXMsIHJlbW92ZWQgbmVpZ2hib3JzLCBhbmQgYWN0aW9uIGNvdW50cyBwZXIgc2VlZC4gVGhlIHBvbGljeSBwcm90ZWN0cyB0aGUgZmlyc3QgdHdvIHBvc2l0aW9ucyBpbiBlYWNoIGNhbmRpZGF0ZSByb3cgYW5kIGV2ZXJ5IGVkZ2UgaW4gdGhlIHVuaW9uIG9mIHRoZSBzaXggZmlyc3QgdG91cnMuIFRoZSBDIGhvb2sgaW5kZXBlbmRlbnRseSBjaGVja3MgdGhlIGBwcm90ZWN0ZWQuZWRnZXNgIGZpbGUgYmVmb3JlIGFjY2VwdGluZyBzd2Fwcy4gQSBwcm9wb3NlZCBuZXcgbmVpZ2hib3IgaXMgYWJzZW50IGZyb20gdGhlIGxpdmUgKnVuZGlyZWN0ZWQqIGNhbmRpZGF0ZSBncmFwaCBhbmQgZnJvbSBhbGwgc2l4IGZpcnN0IHRvdXJzLiBUaGUgQyBob29rIHJlcGxhY2VzIHRoZSB2aWN0aW0gaW4gdGhlIHNhbWUgcm93IHBvc2l0aW9uLCBsZWF2ZXMgdGhlIHJvdyBsZW5ndGggdW5jaGFuZ2VkLCBhbmQgd3JpdGVzIGBwb3N0LmNhbmRpZGF0ZXNgOyBQeXRob24gdmVyaWZpZXMgdGhlIGV4YWN0IHJvdyBiZWZvcmUgYWNjZXB0aW5nIGFuIGVuZHBvaW50LgoKRWFjaCBhcm0gc3RhcnRzIHNpeCBmcmVzaCBMS0ggcHJvY2Vzc2VzIHdpdGggdGhlIHNhbWUgc2VlZHMuIFRoZSBmb3VyIHBhdGNoZWQgYXJtcyBtdXN0IG1hdGNoIG9uIHNpeCBmaXJzdCB0b3VycywgbmF0aXZlL2xpdmUgY2FuZGlkYXRlIHNuYXBzaG90cywgbW9kZWwgZGVjaXNpb24gYXJyYXksIHNvdXJjZSBnYXRlcywgdmljdGltcywgYW5kIGFjdGlvbiBjYXBhY2l0aWVzIGZvciBhIHBvbGljeSBjb21wYXJpc29uLiBBIGNvbW1vbiB3YWxsIGRlYWRsaW5lIHdpdGhpbiBlYWNoIGhhcmR3YXJlIHByb2ZpbGUgaW5jbHVkZXMgTEtIIHN0YXJ0dXAsIGZpcnN0IHRyaWFsLCBiYXJyaWVyLCBtb2RlbCBpbmZlcmVuY2UsIGFuZCBzdWJzZXF1ZW50IHNlYXJjaC4gVGhlIHByaW1hcnkgZXhwbG9yYXRvcnkgZW5kcG9pbnQgaXMgdGhlIHNob3J0ZXN0IHZhbGlkIEVVQ18yRCB0b3VyIG9mIHNpeCBzZWVkcyBhdCB0aGUgZGVhZGxpbmU7IGVhY2ggaW5zdGFuY2UgaXMgb25lIHVuaXQuIFRoZSBzaXggc2VlZHMgYXJlIG5vdCBzaXggaW5kZXBlbmRlbnQgcmVwbGljYXRlcy4KCiMjIENvbGFiCgpPcGVuIFtgbm90ZWJvb2tzLzE2X0JldGFUb3VyX3YxNXIyX2xpdmVfc3dhcF9waWxvdC5pcHluYmBdKC4uLy4uL25vdGVib29rcy8xNl9CZXRhVG91cl92MTVyMl9saXZlX3N3YXBfcGlsb3QuaXB5bmIpIGluIENvbGFiIFBybyBhbmQgcnVuIGFsbCBjZWxscy4gQ1BVLCBHUFUsIG9yIEhpZ2ggUkFNIHJ1bnRpbWVzIGFyZSBhY2NlcHRlZC4gQ1VEQSBpcyB1c2VkIHdoZW4gYXZhaWxhYmxlOyBDUFUgaXMgdGhlIGZhbGxiYWNrLiBUaGUgbm90ZWJvb2sgcmVjb3JkcyBHUFUsIHZpc2libGUgdkNQVSBjb3VudCBhbmQgQ1BVIG1vZGVsLiBJdCBzZWxlY3RzIDMwMCBzZWNvbmRzIHBlciBhcm0gb24gQ1BVIG9yIHdpdGggZmV3ZXIgdGhhbiBzaXggdkNQVXMsIGFuZCAxMjAgc2Vjb25kcyBvbiBHUFUgd2l0aCBhdCBsZWFzdCBzaXggdkNQVXMuIEVhY2ggcHJvZmlsZSBnZXRzIGEgZGlzdGluY3QgRHJpdmUgZXhwZXJpbWVudCBuYW1lIGFuZCBtdXN0IGJlIGFuYWx5emVkIHNlcGFyYXRlbHkuCgpUaGUgb3JpZ2luYWwgW3YxNSBub3RlYm9va10oLi4vLi4vbm90ZWJvb2tzLzE1X0JldGFUb3VyX3YxNV9saXZlX3N3YXBfcGlsb3QuaXB5bmIpIHJlbWFpbnMgdGhlIGZyb3plbiByZWNvcmQgb2YgdGhlIDEyMC1zZWNvbmQgYXR0ZW1wdC4gVGhlIHIyIG5vdGVib29rIGFkZHMgZGVhZGxpbmUgZGlhZ25vc3RpY3MgYW5kIHNhdmVzIGZhaWxlZCBydW4gZmlsZXMgdG8gRHJpdmUuIElmIDMwMCBzZWNvbmRzIHN0aWxsIGNhbm5vdCByZWFjaCB0aGUgYmFycmllciBhbmQgcmVzdW1lLCB0aGUgYXR0ZW1wdCByZW1haW5zIGluY29tcGxldGUgYW5kIGl0cyByZWNvcmRlZCBwaGFzZSBhbmQgcGVyLXNlZWQgbG9ncyBpZGVudGlmeSB3aGVyZSBpdCBzdG9wcGVkLgoKVGhlIG5vdGVib29rIHN0b3JlcyBlYWNoIGNvbXBsZXRlZCBhcm0gYXRvbWljYWxseSB1bmRlciBgTXlEcml2ZS9CZXRhVG91ci88ZXhwZXJpbWVudF9uYW1lPmAgKGZvciBleGFtcGxlIGB2MTVyMl9saXZlX3N3YXBfcGlsb3RfY3B1MzAwYCkuIE9uIHJlc3RhcnQgaXQgdmVyaWZpZXMgY29tcGxldGVkIGFybXMgYW5kIHNraXBzIHRoZW07IGFuIGludGVycnVwdGVkIGFybSBzdGFydHMgaW4gYSBuZXcgYXR0ZW1wdCBkaXJlY3RvcnkuIEEgY2hhbmdlZCBjb2RlL3NldHRpbmdzL2hhcmR3YXJlIHByb2ZpbGUgcmVxdWlyZXMgYSBuZXcgZXhwZXJpbWVudCBuYW1lLiBUaGUgY29tcGFjdCByZXN1bHRzIFpJUCBkb3dubG9hZHMgdGhyb3VnaCB0aGUgYnJvd3NlciBhZnRlciBjb21wbGV0aW9uLCB3aGlsZSB0aGUgZnVsbCBhdWRpdCBaSVAgc3RheXMgaW4gRHJpdmUuIFRoZSBhdWRpdCBpbmNsdWRlcyB0b3Vycywgc25hcHNob3RzLCBhY3Rpb24gZmlsZXMsIHByb3RlY3RlZCBlZGdlcywgdGVsZW1ldHJ5LCBhbmQgZW1iZWRkZWQgY29kZS4gRmFpbGVkIGF0dGVtcHRzIGFyZSBjb3BpZWQgdG8gYGVycm9ycy9gIHdpdGggdGhlaXIgZGlhZ25vc3RpYyBmaWxlcy4KClJlZ2VuZXJhdGUgdGhlIG5vdGVib29rIHdpdGg6CgpgYGBiYXNoCnB5dGhvbiBleHBlcmltZW50cy92MTVfbGl2ZV9zd2FwL2J1aWxkX2NvbGFiX25vdGVib29rLnB5CmBgYAoKIyMgSW50ZXJwcmV0YXRpb24KCkEgc3dhcCBhcm0gYmVhdGluZyB0aGUgc2hhbSBidXQgbG9zaW5nIHRvIG5hdGl2ZSBQT1BNVVNJQy02IHdvdWxkIG1lYW4gdGhlIGludGVydmVudGlvbiBoYXMgYSBtZWFzdXJhYmxlIHNlYXJjaCBlZmZlY3QgYnV0IGRvZXMgbm90IHJlcGF5IGl0cyBiYXJyaWVyIGFuZCBpbmZlcmVuY2UgY29zdC4gQSB3aW4gb3ZlciBuYXRpdmUgUE9QTVVTSUMtNiBvbiBmb3VyIHJldXNlZCBpbnN0YW5jZXMgaXMgYSBsZWFkIGZvciBhIGZyb3plbiwgZGlzam9pbnQgdGVzdDsgaXQgaXMgbm90IGV2aWRlbmNlIG9mIGdlbmVyYWwgc29sdmVyIHN1cGVyaW9yaXR5LiBDYW5kaWRhdGUgZWRnZSB2aXNpdHMsIGFjY2VwdGVkLXRvdXIgZW50cmllcywgYW5kIGFwcGVhcmFuY2VzIGluIGltcHJvdmVkIGluY3VtYmVudHMgYXJlIG1lY2hhbmlzbSBvYnNlcnZhdGlvbnMsIG5vdCBjYXVzYWwgY3JlZGl0LiBUaGUgdjEyIHNlbGYtY29ycmVjdGlvbiByZXN1bHQgc3VnZ2VzdHMgdGhhdCB0YXJnZXQtdG91ciBlZGdlIHJlY2FsbCBpcyBhIHdlYWsgcHJveHkgZm9yIHNlYXJjaCBnYWluLCBzbyB0aGUgbmV4dCBtb2RlbCBzaG91bGQgYmUgdHJhaW5lZCBvbiBtYXJnaW5hbCBjb250aW51YXRpb24gZ2FpbiBwZXIgdW5pdCB0aW1lIGlmIHRoaXMgcGlsb3Qgd2FycmFudHMgZnVydGhlciB3b3JrLgo=', 'v15_live_swap/build_colab_notebook.py': '...'}
CODE_ROOT = WORK_ROOT / "code"
for rel, encoded in BUNDLED_FILES.items():
    content = base64.b64decode(encoded)
    for dest in (CODE_ROOT / rel, OUTPUT_ROOT / "code" / rel):
        dest.parent.mkdir(parents=True, exist_ok=True)
        if dest.exists():
            assert dest.read_bytes() == content, f"Deney kodu değişmiş: {rel}"
        else:
            dest.write_bytes(content)
CODE_SHA256 = {rel: hashlib.sha256(base64.b64decode(encoded)).hexdigest()
               for rel, encoded in BUNDLED_FILES.items()}
sys.path.insert(0, str(CODE_ROOT / "v15_live_swap"))
import build_search_hook
from test_search_hook import run_smoke as run_swap_hook_smoke
from coordinator import read_tour
from pilot import run_guided_arm, run_native_arm, sha256_file
from policy import POLICY_VERSION
print("Dondurulmuş deney kodu:", CODE_SHA256)


In [ ]:
LKH_ROOT = WORK_ROOT / "lkh"
LKH_ROOT.mkdir(exist_ok=True)
SOURCE_DIR = LKH_ROOT / "LKH-3.0.14"
SOURCE_INFO_PATH = LKH_ROOT / "source_info.json"
if not SOURCE_DIR.exists():
    sources = [
        ("official", "https://webhotel4.ruc.dk/~keld/research/LKH-3/LKH-3.0.14.tgz", None, 12),
        ("pinned_mirror", "https://raw.githubusercontent.com/ZhekaiChen/ALCO/main/third_party/LKH3/LKH-3.0.14.tar", LKH_TAR_SHA256, 90),
    ]
    last_error = None
    for label, url, expected_sha, timeout_s in sources:
        archive = LKH_ROOT / ("LKH-3.0.14.tgz" if url.endswith(".tgz") else "LKH-3.0.14.tar")
        try:
            request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(request, timeout=timeout_s) as response, archive.open("wb") as out:
                shutil.copyfileobj(response, out)
            digest = hashlib.sha256(archive.read_bytes()).hexdigest()
            if expected_sha:
                assert digest == expected_sha, f"LKH arşiv SHA256 uyuşmuyor: {digest}"
            with tarfile.open(archive, "r:*") as tar:
                try:
                    tar.extractall(LKH_ROOT, filter="data")
                except TypeError:
                    tar.extractall(LKH_ROOT)
            assert SOURCE_DIR.exists()
            actual_source_sha = hashlib.sha256((SOURCE_DIR / "SRC/FindTour.c").read_bytes()).hexdigest()
            assert actual_source_sha == LKH_FINDTOUR_SHA256, f"LKH 3.0.14 kaynağı beklenenden farklı: {actual_source_sha}"
            SOURCE_INFO_PATH.write_text(json.dumps({"label": label, "url": url, "archive_sha256": digest}, indent=2))
            break
        except Exception as exc:
            last_error = exc
            if SOURCE_DIR.exists():
                shutil.rmtree(SOURCE_DIR)
    else:
        raise RuntimeError(f"LKH indirilemedi: {last_error}")
assert hashlib.sha256((SOURCE_DIR / "SRC/FindTour.c").read_bytes()).hexdigest() == LKH_FINDTOUR_SHA256
for filename, expected in build_search_hook.UPSTREAM_SHA256.items():
    assert hashlib.sha256((SOURCE_DIR / "SRC" / filename).read_bytes()).hexdigest() == expected, filename
if not SOURCE_INFO_PATH.exists():
    cached = next((p for p in (LKH_ROOT / "LKH-3.0.14.tar", LKH_ROOT / "LKH-3.0.14.tgz") if p.exists()), None)
    assert cached is not None, "LKH kaynak kayıt dosyası eksik; kaynak arşiv bulunamadı"
    SOURCE_INFO_PATH.write_text(json.dumps({"label": "recovered_cache", "url": "local_cache",
                                            "archive_sha256": hashlib.sha256(cached.read_bytes()).hexdigest()}, indent=2))
NATIVE_LKH = SOURCE_DIR / "LKH"
if not NATIVE_LKH.exists():
    with (LKH_ROOT / "native_build.log").open("w") as build_log:
        subprocess.run(["make", "-j", str(min(6, CPU_COUNT))], cwd=SOURCE_DIR, check=True,
                       stdout=build_log, stderr=subprocess.STDOUT)
PATCHED_DIR = LKH_ROOT / "LKH-3.0.14-BetaTour-v15r2-swap"
PATCHED_LKH = PATCHED_DIR / "LKH"
if not PATCHED_LKH.exists():
    if PATCHED_DIR.exists():
        shutil.rmtree(PATCHED_DIR)
    build_search_hook.build(SOURCE_DIR, PATCHED_DIR, jobs=min(6, CPU_COUNT))
patch_manifest = json.loads((PATCHED_DIR / "betatour_build.json").read_text())
assert patch_manifest["upstream_sha256"] == build_search_hook.UPSTREAM_SHA256
assert patch_manifest["search_hook_sha256"] == CODE_SHA256["v15_live_swap/BetaTourSwapHook.h"]
assert patch_manifest["visit_header_sha256"] == CODE_SHA256["v15_live_swap/BetaTourVisit.h"]
assert patch_manifest["binary_sha256"] == sha256_file(PATCHED_LKH)
assert patch_manifest["swap_at_barrier"] is True
run_swap_hook_smoke(PATCHED_LKH)
BUILD_INFO = {"native_sha256": sha256_file(NATIVE_LKH),
              "patched_sha256": sha256_file(PATCHED_LKH),
              "upstream_findtour_sha256": LKH_FINDTOUR_SHA256,
              "action_rule": patch_manifest["action_rule"],
              "candidate_reordering": patch_manifest["upstream_candidate_reordering"]}
origin_path = OUTPUT_ROOT / "code" / "lkh_source_info.json"
if not origin_path.exists():
    origin_path.write_text(SOURCE_INFO_PATH.read_text())
print("LKH 3.0.14 derlendi:", {k: v for k, v in BUILD_INFO.items() if k != "candidate_reordering"})
print("LKH aday sırası yeniden düzenleme kuralı (kaynak denetimi):",
      json.dumps(BUILD_INFO["candidate_reordering"], indent=1))


In [ ]:
def write_json_atomic(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False, allow_nan=False))
    os.replace(tmp, path)

MANIFEST = {"schema": 1, "profile": "four_reused_10k_development_instances",
            "experiment_name": EXPERIMENT_NAME, "arms": ARMS, "seeds": SEEDS,
            "instance_specs": INSTANCE_SPECS, "xy_hashes": EXPECTED_XY_SHA256,
            "policy_version": POLICY_VERSION, "swap_fraction": SWAP_FRACTION,
            "graph_k": GRAPH_K, "repo_commit": REPO_COMMIT,
            "checkpoint_sha256": CHECKPOINT_SHA256,
            "hardware_profile": HARDWARE_PROFILE, "code_sha256": CODE_SHA256,
            "build": BUILD_INFO,
            "analysis_plan": "Exploratory 4-instance swap pilot; best-of-six valid cost at a common per-arm wall deadline selected by hardware profile; no independent superiority claim."}
manifest_path = OUTPUT_ROOT / "manifest.json"
if manifest_path.exists():
    saved = json.loads(manifest_path.read_text())
    assert saved == MANIFEST, "Drive'daki pilot kodu, ayarı veya GPU/vCPU profili farklı; yeni deney adı kullanın."
else:
    write_json_atomic(manifest_path, MANIFEST)

# Exercise model inference, the patched first-trial barrier, and native parity.
warm_xy = T.to_int_coords(T.gen_instance(100, "uniform", np.random.default_rng(9090)))
warm_succ = np.stack([np.roll(np.arange(100), -1)] * 6, axis=1).astype(np.int32)
M.predict_tourfold(MODEL, warm_xy, GRAPH_K, DEVICE,
                   ens_succ=warm_succ, ens_cost=np.ones(6))
warm_problem = WORK_ROOT / "warmup.tsp"
T.write_tsp(warm_problem, warm_xy)
warm_dir = WORK_ROOT / "warmup" / uuid.uuid4().hex
warm_dir.parent.mkdir(exist_ok=True)
warm = run_guided_arm("sham", PATCHED_LKH, warm_problem, warm_xy, warm_dir,
                      SEEDS, max(45.0, 90.0 / CPU_COUNT), MODEL, DEVICE, T, M,
                      fraction=SWAP_FRACTION, graph_k=GRAPH_K,
                      decision_cache_dir=WORK_ROOT / "warmup_cache",
                      model_identity=CHECKPOINT_SHA256)
assert warm["policy_audit"]["selected_actions_per_seed"] == [0] * 6
def edge_set(tour):
    tour = np.asarray(tour, dtype=np.int32)
    return set(zip(np.minimum(tour, np.roll(tour, -1)),
                   np.maximum(tour, np.roll(tour, -1))))
for seed in SEEDS:
    native = T.run_lkh(str(NATIVE_LKH), str(warm_problem),
                       {"CANDIDATE_SET_TYPE": "POPMUSIC", "MAX_CANDIDATES": 6,
                        "SUBGRADIENT": "NO", "SEED": seed, "MAX_TRIALS": 1,
                        "TIME_LIMIT": 30, "STOP_AT_OPTIMUM": "NO"},
                       str(WORK_ROOT / "warmup" / f"native_{seed}_{uuid.uuid4().hex}"), timeout=30)
    first = np.asarray(read_tour(warm_dir / f"seed_{seed}" / "first.tour")) - 1
    assert edge_set(native["tour"]) == edge_set(first), f"İlk tur uyuşmuyor: {seed}"
print("Model, yama, korunan kenarlar ve ilk tur eşitliği doğrulandı.")


In [ ]:
INSTANCES = []
for group, n, count, distribution, generator_seed, budget_s in INSTANCE_SPECS:
    rng = np.random.default_rng(generator_seed)
    for index in range(count):
        xy = T.to_int_coords(T.gen_instance(n, distribution, rng))
        name = f"augment_{group}_{index:03d}"
        digest = hashlib.sha256(np.ascontiguousarray(xy).tobytes()).hexdigest()
        assert digest == EXPECTED_XY_SHA256[name], f"Eski v8 koordinatları uyuşmuyor: {name}"
        INSTANCES.append({"name": name, "group": group, "n": n,
                          "budget_s": budget_s, "xy": xy, "xy_sha256": digest})
INSTANCES.sort(key=lambda item: (item["budget_s"], item["name"]))
assert len(INSTANCES) == 4
def arm_order(name):
    return sorted(ARMS, key=lambda arm: hashlib.sha256((name + ":" + arm).encode()).hexdigest())
print("Pilot:", len(INSTANCES), "örnek,", len(ARMS), "kol; kesintisiz tek kol bütçesi",
      sum(item["budget_s"] for item in INSTANCES) / 3600, "saat")


In [ ]:
"""v15 live-swap execution cell. Completed arms are immutable and resumable."""


def arm_record_path(inst, arm):
    return OUTPUT_ROOT / "instances" / inst["name"] / arm / "result.json"


def validate_saved(inst, arm, record):
    assert record["complete"] and record["arm"] == arm and record["instance"] == inst["name"]
    assert record["xy_sha256"] == inst["xy_sha256"] and record["seeds"] == SEEDS
    assert record["budget_s"] == inst["budget_s"] and record["build"] == BUILD_INFO
    assert record["code_sha256"] == CODE_SHA256
    assert record["fraction"] == SWAP_FRACTION and record["graph_k"] == GRAPH_K
    assert record["hardware_profile"] == HARDWARE_PROFILE
    assert len(record["seed_costs"]) == len(SEEDS)
    assert min(record["seed_costs"]) == record["best_cost"]
    if arm == "native_pop6":
        assert set(record["final_tour_sources"]) == {str(seed) for seed in SEEDS}
    else:
        assert len(record["first_tour_hashes"]) == len(SEEDS)
        assert len(record["live_candidate_hashes"]) == len(SEEDS)
        assert len(record["native_candidate_hashes"]) == len(SEEDS)
        assert len(record["protection_sha256"]) == 64
        audit = record["policy_audit"]
        assert len(audit["gate_sha256_per_seed"]) == len(SEEDS)
        assert audit["seed_ids"] == SEEDS
        assert audit["policy_version"] == POLICY_VERSION
        assert audit["decision_basis"] in ("fresh_inference", "cached_inference")
        if arm == "sham":
            assert audit["selected_actions_per_seed"] == [0] * len(SEEDS)
        else:
            assert audit["selected_actions_per_seed"] == audit["guided_actions_per_seed"]
        assert audit["selected_unique_undirected_edges_per_seed"] == audit["selected_actions_per_seed"]
    return record


def save_progress(state="running", last=None):
    completed = sum(arm_record_path(inst, arm).exists() for inst in INSTANCES for arm in ARMS)
    write_json_atomic(OUTPUT_ROOT / "progress.json",
                      {"state": state, "completed_arms": completed,
                       "total_arms": len(INSTANCES) * len(ARMS), "last": last,
                       "updated_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())})


new_arms = 0
save_progress()
for inst in INSTANCES:
    instance_dir = OUTPUT_ROOT / "instances" / inst["name"]
    instance_dir.mkdir(parents=True, exist_ok=True)
    coord_path = instance_dir / "xy.npz"
    if not coord_path.exists():
        temp = instance_dir / "xy.tmp.npz"
        np.savez_compressed(temp, xy=inst["xy"])
        os.replace(temp, coord_path)
    with np.load(coord_path, allow_pickle=False) as saved_xy:
        assert hashlib.sha256(np.ascontiguousarray(saved_xy["xy"]).tobytes()).hexdigest() == inst["xy_sha256"]
    local_problem = WORK_ROOT / "problems" / (inst["name"] + ".tsp")
    local_problem.parent.mkdir(exist_ok=True)
    T.write_tsp(local_problem, inst["xy"])
    for arm in arm_order(inst["name"]):
        record_path = arm_record_path(inst, arm)
        if record_path.exists():
            validate_saved(inst, arm, json.loads(record_path.read_text()))
            print("ATLA:", inst["name"], arm, flush=True)
            continue
        if MAX_NEW_ARMS is not None and new_arms >= MAX_NEW_ARMS:
            break
        attempt_id = uuid.uuid4().hex
        local_run = WORK_ROOT / "runs" / inst["name"] / arm / attempt_id
        local_run.parent.mkdir(parents=True, exist_ok=True)
        try:
            print("BAŞLA:", inst["name"], arm, "bütçe", inst["budget_s"], "sn", flush=True)
            if arm.startswith("native_"):
                raw = run_native_arm(arm, NATIVE_LKH, local_problem, inst["xy"],
                                     local_run, SEEDS, inst["budget_s"], T)
            else:
                # Arms whose model input is bitwise identical decide from the
                # first stored model output; every arm still runs inference.
                raw = run_guided_arm(arm, PATCHED_LKH, local_problem, inst["xy"],
                                     local_run, SEEDS, inst["budget_s"],
                                     MODEL, DEVICE, T, M,
                                     fraction=SWAP_FRACTION, graph_k=GRAPH_K,
                                     decision_cache_dir=instance_dir / "decision_cache",
                                     model_identity=CHECKPOINT_SHA256)
            audit_rel = f"instances/{inst['name']}/{arm}/{attempt_id}"
            audit_dest = OUTPUT_ROOT / audit_rel
            shutil.copytree(local_run, audit_dest)
            record = {"schema": 1, "complete": True, "instance": inst["name"],
                      "group": inst["group"], "n": inst["n"],
                      "xy_sha256": inst["xy_sha256"], "arm": arm,
                      "seeds": SEEDS, "budget_s": inst["budget_s"],
                      "seed_costs": raw["seed_costs"], "best_cost": raw["best_cost"],
                      "wall_elapsed_s": raw["wall_elapsed_s"],
                      "phase_timings_s": raw.get("phase_timings_s"),
                      "remaining_at_policy_end_s": raw.get("remaining_at_policy_end_s"),
                      "start_epoch_s": raw["start_epoch_s"],
                      "deadline_epoch_s": raw["deadline_epoch_s"] if "deadline_epoch_s" in raw else raw["start_epoch_s"] + inst["budget_s"],
                      "first_costs": raw.get("first_costs"),
                      "first_tour_hashes": raw.get("first_tour_hashes"),
                      "live_candidate_hashes": raw.get("live_candidate_hashes"),
                      "native_candidate_hashes": raw.get("native_candidate_hashes"),
                      "protection_sha256": raw.get("protection_sha256"),
                      "policy_audit": raw.get("policy_audit"),
                      "guided_improvement_events": raw.get("guided_improvement_events"),
                      "native_improvement_events": raw.get("native_improvement_events"),
                      "final_tour_sources": raw.get("final_tour_sources"),
                      "event_flags": raw.get("event_flags"),
                      "decision_cache_files_written": raw.get("decision_cache_files_written"),
                      "fraction": SWAP_FRACTION, "graph_k": GRAPH_K,
                      "build": BUILD_INFO, "code_sha256": CODE_SHA256,
                      "hardware": HARDWARE, "hardware_profile": HARDWARE_PROFILE,
                      "audit_relpath": audit_rel,
                      "attempt_id": attempt_id,
                      "completed_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
            validate_saved(inst, arm, record)
            write_json_atomic(record_path, record)
            new_arms += 1
            save_progress(last=inst["name"] + "/" + arm)
            print("BİTTİ:", inst["name"], arm, "en iyi", raw["best_cost"], flush=True)
        except Exception as exc:
            failure_dir = OUTPUT_ROOT / "errors" / (inst["name"] + "_" + arm + "_" + attempt_id)
            failure_dir.mkdir(parents=True, exist_ok=True)
            audit_copy_error = None
            if local_run.exists():
                try:
                    shutil.copytree(local_run, failure_dir / "audit")
                except Exception as copy_exc:
                    audit_copy_error = repr(copy_exc)
            write_json_atomic(failure_dir / "error.json",
                              {"instance": inst["name"], "arm": arm,
                               "attempt_id": attempt_id, "budget_s": inst["budget_s"],
                               "hardware_profile": HARDWARE_PROFILE,
                               "error": repr(exc), "audit_copy_error": audit_copy_error,
                               "utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())})
            print("Başarısız deneme kayıtları:", failure_dir, flush=True)
            save_progress(state="error", last=inst["name"] + "/" + arm)
            raise
    if MAX_NEW_ARMS is not None and new_arms >= MAX_NEW_ARMS:
        break

completed_count = sum(arm_record_path(inst, arm).exists() for inst in INSTANCES for arm in ARMS)
save_progress(state="complete" if completed_count == len(INSTANCES) * len(ARMS) else "partial")
print(f"Tamamlanan kol: {completed_count}/{len(INSTANCES) * len(ARMS)}. "
      f"Bu oturumdaki yeni kol: {new_arms}")


In [ ]:
"""v15 live-swap exploratory analysis on paired development instances.

Only complete same-hardware instances contribute to endpoint comparisons.
Policy comparisons additionally require exact first tours, candidate states,
model decision arrays, source gates, victims and action capacities.  The four
instances were examined in previous runs, so p-values are descriptive only.
"""

from math import comb


def read_key_value_metrics(path):
    values = {}
    for line in Path(path).read_text().splitlines():
        words = line.split()
        if len(words) != 2 or words[0] in values:
            raise ValueError(f"Bozuk telemetri satırı: {path}: {line!r}")
        values[words[0]] = float(words[1]) if "." in words[1] else int(words[1])
    return values


def exact_sign_p(wins, losses):
    n, k = wins + losses, min(wins, losses)
    return 1.0 if n == 0 else min(1.0, 2.0 * sum(comb(n, j) for j in range(k + 1)) / 2 ** n)


def compare(frame, treatment, control, basis, column="{}"):
    if frame.empty:
        return None
    t = frame[column.format(treatment)].to_numpy(dtype=np.float64)
    c = frame[column.format(control)].to_numpy(dtype=np.float64)
    rel = 100.0 * (t - c) / c
    wins, losses = int((rel < 0).sum()), int((rel > 0).sum())
    return {"treatment": treatment, "control": control, "basis": basis,
            "n_instances": len(rel), "wins": wins, "losses": losses,
            "ties": int((rel == 0).sum()),
            "mean_relative_pct": float(rel.mean()),
            "median_relative_pct": float(np.median(rel)),
            "exact_sign_p_descriptive": exact_sign_p(wins, losses)}


patched_arms = ["sham", "random_swap", "nearest_swap", "beta_swap"]
guided_arms = patched_arms[1:]
rows, seed_rows, mechanism_rows = [], [], []
summary_dir = OUTPUT_ROOT / "summary"
summary_dir.mkdir(exist_ok=True)

for inst in INSTANCES:
    records = {}
    for arm in ARMS:
        path = arm_record_path(inst, arm)
        if path.exists():
            records[arm] = validate_saved(inst, arm, json.loads(path.read_text()))
    if len(records) != len(ARMS):
        continue
    audits = {arm: records[arm]["policy_audit"] for arm in patched_arms}
    row = {"instance": inst["name"], "group": inst["group"],
           "n": inst["n"], "budget_s": inst["budget_s"],
           "mixed_hardware": len({json.dumps(records[arm]["hardware_profile"], sort_keys=True)
                                  for arm in ARMS}) != 1}
    for arm in ARMS:
        row[arm] = records[arm]["best_cost"]
        row["mean6_" + arm] = float(np.mean(records[arm]["seed_costs"]))
    row["first_tours_match"] = all(
        len({records[arm]["first_tour_hashes"][i] for arm in patched_arms}) == 1
        for i in range(len(SEEDS)))
    row["raw_first_tours_match"] = all(
        len({audits[arm]["first_tour_raw_sha256"][i] for arm in patched_arms}) == 1
        for i in range(len(SEEDS)))
    row["native_candidates_match"] = all(
        len({records[arm]["native_candidate_hashes"][i] for arm in patched_arms}) == 1
        for i in range(len(SEEDS)))
    row["live_candidates_match"] = all(
        len({records[arm]["live_candidate_hashes"][i] for arm in patched_arms}) == 1
        for i in range(len(SEEDS)))
    row["protected_edges_match"] = len({records[arm]["protection_sha256"]
                                         for arm in patched_arms}) == 1
    row["gates_match"] = all(
        len({audits[arm]["gate_sha256_per_seed"][i] for arm in patched_arms}) == 1
        for i in range(len(SEEDS)))
    row["victims_match"] = all(
        len({audits[arm]["victim_sha256_per_seed"][i] for arm in patched_arms}) == 1
        for i in range(len(SEEDS)))
    row["model_predictions_match"] = len({audits[arm]["model_probability_sha256"]
                                           for arm in patched_arms}) == 1
    row["planned_actions_match"] = len({
        json.dumps(audits[arm]["all_guided_action_sha256_per_seed"], sort_keys=True)
        for arm in patched_arms}) == 1
    row["capacities_match"] = all(
        len({audits[arm]["selected_actions_per_seed"][i] for arm in guided_arms}) == 1
        for i in range(len(SEEDS)))
    row["matched_policy_state"] = all(row[key] for key in (
        "first_tours_match", "raw_first_tours_match", "native_candidates_match",
        "live_candidates_match", "protected_edges_match", "gates_match",
        "victims_match", "model_predictions_match", "planned_actions_match",
        "capacities_match"))
    rows.append(row)

    for i, seed in enumerate(SEEDS):
        seed_rows.append({"instance": inst["name"], "seed": seed,
                          **{arm: records[arm]["seed_costs"][i] for arm in ARMS}})
    for arm in patched_arms:
        for i, seed in enumerate(SEEDS):
            base = OUTPUT_ROOT / records[arm]["audit_relpath"] / f"seed_{seed}"
            metrics = read_key_value_metrics(base / "telemetry.txt")
            applied = read_key_value_metrics(base / "applied.stats")
            selected = int(audits[arm]["selected_actions_per_seed"][i])
            assert int(metrics["action_count"]) == selected
            assert int(applied["swapped"]) == selected
            assert int(applied["rejected"]) == 0
            assert int(applied["row_length_preserved"]) == 1
            mechanism_rows.append({"instance": inst["name"], "arm": arm,
                                   "seed": seed, "selected_actions": selected,
                                   "barrier_wait_s": float(applied["wait_wall_s"]),
                                   "telemetry_age_at_end_s": max(0.0, min(
                                       float(records[arm]["deadline_epoch_s"]),
                                       float(records[arm]["start_epoch_s"]) +
                                       float(records[arm]["wall_elapsed_s"])) -
                                       float(metrics["telemetry_snapshot_epoch_s"])),
                                   **metrics})

table = pd.DataFrame(rows)
seed_table = pd.DataFrame(seed_rows)
mechanism_table = pd.DataFrame(mechanism_rows)
table.to_csv(summary_dir / "complete_instances.csv", index=False)
seed_table.to_csv(summary_dir / "seed_costs.csv", index=False)
mechanism_table.to_csv(summary_dir / "swap_mechanism.csv", index=False)

same_hardware = table.loc[~table["mixed_hardware"]].copy() if not table.empty else table
matched = same_hardware.loc[same_hardware["matched_policy_state"]].copy() if not same_hardware.empty else same_hardware
comparisons = []
for control in ("native_pop6", "sham", "nearest_swap", "random_swap"):
    basis = "same hardware, best-of-six endpoint"
    frame = same_hardware if control == "native_pop6" else matched
    result = compare(frame, "beta_swap", control, basis)
    if result:
        comparisons.append(result)
    result = compare(frame, "beta_swap", control, "descriptive mean-of-six", "mean6_{}")
    if result:
        comparisons.append(result)
comparison_table = pd.DataFrame(comparisons)
comparison_table.to_csv(summary_dir / "exploratory_comparisons.csv", index=False)

mechanism_summary = pd.DataFrame()
if not mechanism_table.empty:
    measures = [name for name in (
        "selected_actions", "action_candidate_enumerations",
        "action_accepted_edge_entries", "action_incumbent_entries",
        "trials_completed_after_action", "total_candidate_enumerations")
        if name in mechanism_table.columns]
    mechanism_summary = mechanism_table.groupby("arm", as_index=False)[measures].sum()
mechanism_summary.to_csv(summary_dir / "swap_mechanism_totals.csv", index=False)

write_json_atomic(summary_dir / "pilot_note.json", {
    "completed_instances": len(table), "planned_instances": len(INSTANCES),
    "same_hardware_instances": len(same_hardware),
    "matched_policy_instances": len(matched),
    "interpretation": "Reused-case exploratory pilot. Instance is the analysis unit; seeds are not independent. Sign p-values are descriptive. Native POP6 is the uninterrupted practical benchmark. Directed row length is fixed only at the swap barrier; LKH may append or reorder candidates later. Reverse-row presence of evicted edges can leave the undirected graph size unchanged or increase it. No superiority claim before disjoint holdout."})

print("Tamamlanan örnek:", len(table), "/", len(INSTANCES),
      "; eşlenmiş politika örneği:", len(matched))
display(table)
display(comparison_table)
display(mechanism_summary)


In [ ]:
def checked_zip(name, include_audit):
    local = WORK_ROOT / name
    temp = WORK_ROOT / (name + ".tmp")
    files = []
    with zipfile.ZipFile(temp, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6, allowZip64=True) as archive:
        for path in sorted(OUTPUT_ROOT.rglob("*")):
            if not path.is_file() or path.suffix == ".zip" or path.name.endswith(".tmp") or path.name.endswith(".tmp.npz"):
                continue
            rel = path.relative_to(OUTPUT_ROOT)
            if not include_audit and ("policy_audit" in rel.parts or "decision_cache" in rel.parts or path.name in
                                      {"xy.npz", "native.candidates", "live.candidates", "post.candidates", "first.tour", "best.tour", "action.txt", "protected.edges", "stdout.txt", "stderr.txt", "policy_arrays.npz"}):
                continue
            archive.write(path, str(rel), compress_type=zipfile.ZIP_STORED if path.suffix == ".npz" else zipfile.ZIP_DEFLATED)
            files.append(str(rel))
    with zipfile.ZipFile(temp) as check:
        assert check.testzip() is None and len(check.namelist()) == len(files)
    os.replace(temp, local)
    target = OUTPUT_ROOT / "archives" / name
    tmp_target = target.with_name(target.name + ".tmp")
    shutil.copy2(local, tmp_target)
    os.replace(tmp_target, target)
    assert sha256_file(local) == sha256_file(target)
    return {"local": str(local), "drive": str(target), "sha256": sha256_file(local),
            "bytes": local.stat().st_size, "files": len(files)}

complete = len(rows) == len(INSTANCES)
audit_zip = checked_zip(EXPERIMENT_NAME + "_audit.zip", True) if complete else None
results_zip = checked_zip(EXPERIMENT_NAME + "_results.zip", False)
write_json_atomic(OUTPUT_ROOT / "archives" / "archive_info.json",
                  {"complete": complete, "results": results_zip, "audit": audit_zip})
print("Sonuç ZIP'i:", results_zip)
if complete:
    print("Tam denetim ZIP'i Drive'da:", audit_zip)
    from google.colab import files
    files.download(results_zip["local"])
else:
    print(f"Ara sonuçlar Drive'da; tamamlanan örnek {len(rows)}/{len(INSTANCES)}. Yeniden çalıştırınca devam eder.")
